# Analytic Knot Fields and Deformations

Use this notebook when your starting object is a **knot/link type, Artin braid word, or analytic complex field** rather than a pre-existing embedded graph. It shows how to construct fields with KnottedGraph, validate numerical tubes, extract spatial graphs, and then use the ordinary projection and Yamada-polynomial tools.

The main route is

$$
\boxed{\text{Artin braid word}} \rightarrow f(u,v,\bar v) \rightarrow F(x,y,z) \rightarrow \{|F|\le\epsilon\} \rightarrow \text{spatial graph} \rightarrow \text{PD code} \rightarrow \Upsilon.
$$

Start with the early sections for a first user-level run. The later gallery, convergence, and figure-export sections are optional templates for heavier exploratory or publication-grade workflows.

Read the diagnostics as part of the API, not as decoration: a braid polynomial can be well-defined while a finite Fourier approximation, finite grid, or chosen tube radius still needs validation before it supports a topology claim.


## 0. Installation and runtime controls

Run from a checkout with the knot-field extras installed. For level-set extraction:

```bash
pip install -e ".[knot-fields,notebook]"
```

The quick settings are intentionally modest. Increase them for final scientific runs.

In [ ]:
import ast
import dataclasses
import hashlib
import json
import pickle
import time
from collections import Counter
from pathlib import Path

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

from pathlib import Path
import sys

import knotted_graph
from knotted_graph.invariants.yamada.native import (
    native_available,
    native_import_error,
)

print("Python executable:", sys.executable)
print("KnottedGraph:", Path(knotted_graph.__file__).resolve())
print("Native Yamada backend:", native_available())
print("Native import error:", native_import_error())

from knotted_graph.inputs import (
    KnotFunction, KnotFunctionPath,
    available_knot_names, get_knot_entry,
    infer_braid_strands, braid_permutation, braid_component_count,
    geometric_braid_roots, braid_to_semiholomorphic,
    inverse_stereographic_s3, sample_s3,
)
from knotted_graph.applications.knot_deformation import (
    KnotDeformationRecord, KnotDeformationScan, KnotDeformationScanResult,
)
from knotted_graph.core.embedding import remove_leaf_nodes, simplify_edges, smooth_edges

QUICK = True
PAPER_FIGURE_MODE = True
FAST_INTERACTIVE = False if PAPER_FIGURE_MODE else True
SAVE_PAPER_FIGURES = True

GRID_DIM = 80 if PAPER_FIGURE_MODE else (44 if FAST_INTERACTIVE else (56 if QUICK else 96))
CONVERGENCE_DIMS = (56, 64, 80) if PAPER_FIGURE_MODE else ((32, 40, 44) if FAST_INTERACTIVE else ((40, 48, 56) if QUICK else (64, 96, 128)))

RUN_FULL_SYMBOLIC_R3 = False
RUN_SYMBOLIC_SIMPLIFY = False
RUN_YAMADA = True
RUN_DEFORMATION_SCAN = True if PAPER_FIGURE_MODE else (False if FAST_INTERACTIVE else True)
RUN_TOPOLOGY_PHASE_SCAN = True if PAPER_FIGURE_MODE else (False if FAST_INTERACTIVE else True)
RUN_TOPOLOGY_CONVERGENCE_AUDIT = True if PAPER_FIGURE_MODE else False

RUN_USE_CASE_GALLERY = True
RUN_GALLERY_COMPILER = True
RUN_GALLERY_DIAGNOSTICS = True if PAPER_FIGURE_MODE else (False if FAST_INTERACTIVE else True)
RUN_GALLERY_PHASE_ATLAS = True if PAPER_FIGURE_MODE else (False if FAST_INTERACTIVE else True)
RUN_GALLERY_CONVERGENCE = False
RUN_GALLERY_GRAPH_EXTRACTION = True if PAPER_FIGURE_MODE else False
RUN_PAPER_FIGURE_EXPORT = True if PAPER_FIGURE_MODE else False

GALLERY_DIM = 96 if PAPER_FIGURE_MODE else (28 if FAST_INTERACTIVE else (36 if QUICK else 72))
GALLERY_PHASE_DIM = 160 if PAPER_FIGURE_MODE else (24 if FAST_INTERACTIVE else (32 if QUICK else 64))
GALLERY_CONVERGENCE_DIMS = (56, 64, 80) if PAPER_FIGURE_MODE else ((24, 28, 32) if FAST_INTERACTIVE else ((32, 40, 48) if QUICK else (64, 96, 128)))
GALLERY_RADII = np.linspace(0.10, 0.40, 13) if PAPER_FIGURE_MODE else (np.array([0.08, 0.12, 0.16, 0.20]) if FAST_INTERACTIVE else np.array([0.06, 0.08, 0.10, 0.12, 0.16, 0.20, 0.25]))
GALLERY_PHASE_RADII = np.linspace(0.20, 0.34, 29) if PAPER_FIGURE_MODE else (np.array([0.12, 0.18]) if FAST_INTERACTIVE else np.array([0.10, 0.15, 0.20]))
GALLERY_LAMBDAS = np.linspace(0.0, 1.0, 49 if PAPER_FIGURE_MODE else (3 if FAST_INTERACTIVE else (5 if QUICK else 9)))
TOPOLOGY_PHASE_LAMBDAS = np.linspace(0.0, 1.0, 25) if PAPER_FIGURE_MODE else GALLERY_LAMBDAS
TOPOLOGY_PHASE_RADII = np.linspace(0.10, 1.28, 35) if PAPER_FIGURE_MODE else GALLERY_PHASE_RADII
TOPOLOGY_PHASE_DIM = 144 if PAPER_FIGURE_MODE else GALLERY_PHASE_DIM
TOPOLOGY_SPAN = ((-10.0, 10.0),) * 3 if PAPER_FIGURE_MODE else ((-4.0, 4.0),) * 3

def _is_knotted_graph_repo(candidate: Path) -> bool:
    return (candidate / "src" / "knotted_graph").is_dir() and (candidate / "User_guide" / "applications").is_dir()


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if _is_knotted_graph_repo(candidate):
            return candidate
    raise RuntimeError("Could not find the KnottedGraph repository root.")


ROOT = find_repo_root()
APPLICATION_DIR = ROOT / "User_guide" / "applications"
APPLICATION_RESULTS_DIR = APPLICATION_DIR / "results"
PAPER_FIGURE_DIR = APPLICATION_RESULTS_DIR / "04_analytic_knot_fields" / "figures"
PAPER_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PHASE_CACHE_DIR = APPLICATION_RESULTS_DIR / "04_analytic_knot_fields_cache"
PHASE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

PAPER_SURFACE_DIM = 128 if PAPER_FIGURE_MODE else GALLERY_DIM
PAPER_STRIP_DIM = 128 if PAPER_FIGURE_MODE else GALLERY_PHASE_DIM
PAPER_STRIP_LAMBDAS = np.linspace(0.0, 1.0, 9 if PAPER_FIGURE_MODE else len(GALLERY_LAMBDAS))
YAMADA_NUM_ROTATION_SAMPLES = 32 if PAPER_FIGURE_MODE else 8
YAMADA_OPTIONS = {
    "num_rotation_samples": YAMADA_NUM_ROTATION_SAMPLES,
    "n_jobs": 1,
    "method": "negami",
    "crossing_warning_threshold": 20,
}
YAMADA_GRAPH_SMOOTH_EPS = 0.4
YAMADA_LINK_CORE_SMOOTH_EPS = 0.1
YAMADA_EMBEDDED_BRANCH_EDGE_LIMIT = 3
VALIDATE_YAMADA_TUBES = True
YAMADA_REQUIRE_LINK_CORE = False
S3_VERTEX_BOUND_MARGIN = 1e-10
PHASE_SCAN_CACHE_VERSION = "yamada_compactified_s3_vertex_bound_v11"
TOPOLOGY_SCAN_CACHE_VERSION = "levelset_topology_wide_span_v2_extended_pole"

print("PAPER_FIGURE_MODE =", PAPER_FIGURE_MODE)
print("FAST_INTERACTIVE =", FAST_INTERACTIVE)
print("GRID_DIM =", GRID_DIM)
print("RUN_YAMADA =", RUN_YAMADA)
print("GALLERY_PHASE_DIM =", GALLERY_PHASE_DIM)
print("GALLERY_LAMBDAS =", len(GALLERY_LAMBDAS))
print("GALLERY_PHASE_RADII =", (float(GALLERY_PHASE_RADII[0]), float(GALLERY_PHASE_RADII[-1]), len(GALLERY_PHASE_RADII)))
print("TOPOLOGY_PHASE_RADII =", (float(TOPOLOGY_PHASE_RADII[0]), float(TOPOLOGY_PHASE_RADII[-1]), len(TOPOLOGY_PHASE_RADII)))
print("TOPOLOGY_PHASE_DIM =", TOPOLOGY_PHASE_DIM)
print("TOPOLOGY_SPAN =", TOPOLOGY_SPAN)
print("YAMADA_GRAPH_SMOOTH_EPS =", YAMADA_GRAPH_SMOOTH_EPS)
print("YAMADA_LINK_CORE_SMOOTH_EPS =", YAMADA_LINK_CORE_SMOOTH_EPS)
print("YAMADA_EMBEDDED_BRANCH_EDGE_LIMIT =", YAMADA_EMBEDDED_BRANCH_EDGE_LIMIT)
print("YAMADA_REQUIRE_LINK_CORE =", YAMADA_REQUIRE_LINK_CORE)
print("S3_VERTEX_BOUND_MARGIN =", S3_VERTEX_BOUND_MARGIN)
print("CONVERGENCE_DIMS =", CONVERGENCE_DIMS)
print("RUN_FULL_SYMBOLIC_R3 =", RUN_FULL_SYMBOLIC_R3)
print("RUN_DEFORMATION_SCAN =", RUN_DEFORMATION_SCAN)
print("RUN_TOPOLOGY_PHASE_SCAN =", RUN_TOPOLOGY_PHASE_SCAN)
print("RUN_TOPOLOGY_CONVERGENCE_AUDIT =", RUN_TOPOLOGY_CONVERGENCE_AUDIT)
print("RUN_GALLERY_DIAGNOSTICS =", RUN_GALLERY_DIAGNOSTICS)
print("RUN_GALLERY_PHASE_ATLAS =", RUN_GALLERY_PHASE_ATLAS)
print("RUN_GALLERY_GRAPH_EXTRACTION =", RUN_GALLERY_GRAPH_EXTRACTION)
print("PAPER_FIGURE_DIR =", PAPER_FIGURE_DIR)


### Figure export helpers
These helpers keep every 3D plot visible, set stable 3D bounds, and save paper figures as both PNG and PDF when `SAVE_PAPER_FIGURES=True`.


In [ ]:
PAPER_SURFACE_COLOR = "#5b8cc0"
PAPER_GRAPH_COLOR = "#c43c39"
PAPER_NODE_COLOR = "#111111"


def save_paper_figure(fig, stem, *, dpi=240):
    if not SAVE_PAPER_FIGURES:
        return []
    PAPER_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    paths = [PAPER_FIGURE_DIR / f"{stem}.png", PAPER_FIGURE_DIR / f"{stem}.pdf"]
    for path in paths:
        fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    print("saved:", ", ".join(str(path) for path in paths))
    return paths


def set_axes_equal_3d(ax, points=None, *, span=None, pad=0.04):
    if span is not None:
        mins = np.array([bounds[0] for bounds in span], dtype=float)
        maxs = np.array([bounds[1] for bounds in span], dtype=float)
    elif points is not None and len(points):
        pts = np.asarray(points, dtype=float)
        mins = np.nanmin(pts, axis=0)
        maxs = np.nanmax(pts, axis=0)
    else:
        mins = np.array([-1.0, -1.0, -1.0])
        maxs = np.array([1.0, 1.0, 1.0])
    center = 0.5 * (mins + maxs)
    radius = 0.5 * np.max(maxs - mins)
    radius = radius * (1.0 + pad) if radius > 0 else 1.0
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)


def style_3d_axis(ax, *, labels=False):
    ax.view_init(elev=22, azim=-55)
    ax.grid(False)
    if labels:
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_zlabel("z")
    else:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.set_alpha(0.0)


def plot_level_surface_mesh(ax, mesh, *, color=PAPER_SURFACE_COLOR, alpha=0.32, max_faces=18000):
    faces = mesh.faces
    step = max(1, int(np.ceil(len(faces) / max_faces)))
    ax.plot_trisurf(
        mesh.vertices[:, 0], mesh.vertices[:, 1], mesh.vertices[:, 2],
        triangles=faces[::step], color=color, alpha=alpha,
        linewidth=0.0, antialiased=True, shade=True,
    )
    return mesh.vertices


def plot_spatial_graph_3d(ax, graph, *, color=PAPER_GRAPH_COLOR, node_color=PAPER_NODE_COLOR, linewidth=2.2):
    plotted = []
    for _, _, _, data in graph.edges(keys=True, data=True):
        pts = np.asarray(data.get("pts", []), dtype=float)
        if pts.size == 0:
            continue
        ax.plot(pts[:, 0], pts[:, 1], pts[:, 2], color=color, lw=linewidth, solid_capstyle="round")
        plotted.append(pts)
    node_pts = np.asarray([data["pos"] for _, data in graph.nodes(data=True) if "pos" in data], dtype=float)
    if node_pts.size:
        ax.scatter(node_pts[:, 0], node_pts[:, 1], node_pts[:, 2], s=20, c=node_color, depthshade=False)
        plotted.append(node_pts)
    if plotted:
        return np.vstack(plotted)
    return np.empty((0, 3))


def field_surface_graph(field, eps, *, span=((-4.0, 4.0),) * 3, dimension=64, require_compact=True):
    sample = field.sample(span=span, dimension=dimension)
    diagnostic = field.diagnose_level(eps, sample=sample, span=span, dimension=dimension)
    mesh = field.level_surface(eps, sample=sample, span=span, dimension=dimension, require_compact=require_compact)
    graph = field.to_spatial_graph(eps, sample=sample, span=span, dimension=dimension)
    return sample, diagnostic, mesh, graph


def plot_field_surface_graph_panel(ax, field, eps, *, title, span=((-4.0, 4.0),) * 3, dimension=64):
    sample, diagnostic, mesh, graph = field_surface_graph(field, eps, span=span, dimension=dimension)
    surface_pts = plot_level_surface_mesh(ax, mesh)
    graph_pts = plot_spatial_graph_3d(ax, graph)
    pts = np.vstack([surface_pts, graph_pts]) if len(graph_pts) else surface_pts
    set_axes_equal_3d(ax, pts)
    style_3d_axis(ax)
    ax.set_title(title, pad=4)
    return {
        "field": field, "epsilon": eps, "sample": sample,
        "diagnostic": diagnostic, "mesh": mesh, "graph": graph,
    }


def short_yamada_label(expr, *, max_chars=96):
    if expr is None:
        return "Yamada unavailable"
    rendered = sp.sstr(sp.factor(expr))
    if len(rendered) <= max_chars:
        return rendered
    return rendered[:max_chars - 1] + "..."


def phase_label_lookup(result):
    lookup = {}
    next_index = {"Y": 1, "G": 1}
    for record in result.records:
        yamada = getattr(record, "yamada", None)
        if yamada is None or record.phase_signature in lookup:
            continue
        if record.phase_signature == "yamada:contractible-vertex":
            lookup[record.phase_signature] = "vertex"
            continue
        prefix = "G" if record.phase_signature.startswith("abstract-yamada:") else "Y"
        if sp.simplify(yamada) == 0:
            lookup[record.phase_signature] = f"{prefix}0"
        else:
            lookup[record.phase_signature] = f"{prefix}{next_index[prefix]}"
            next_index[prefix] += 1
    return lookup


def phase_polynomial_lookup(result):
    lookup = {}
    for record in result.records:
        yamada = getattr(record, "yamada", None)
        if yamada is not None and record.phase_signature not in lookup:
            lookup[record.phase_signature] = short_yamada_label(yamada)
    return lookup


def compact_phase_label(signature, yamada_labels=None):
    yamada_labels = yamada_labels or {}
    if signature.startswith("topology:"):
        if signature == "topology:tube":
            return "knot/link tube"
        if signature == "topology:sphere":
            return "sphere"
        if signature.startswith("topology:higher-genus"):
            return "higher genus"
        if signature in {"topology:fragmented", "topology:multi-component"}:
            return "multi-component"
        if signature == "topology:boundary":
            return "box boundary"
        if signature == "topology:open":
            return "open surface"
        if signature == "topology:noncompact-pole":
            return "noncompact pole"
        if signature.startswith("topology:closed"):
            return signature.replace("topology:closed-", "closed ")
        return signature[len("topology:"):].replace("-", " ")
    if signature == "error:boundary" or "sampling-box boundary" in signature:
        return "boundary"
    if signature == "error:chart-gap":
        return "chart gap"
    if signature.startswith("error:topology"):
        return "invalid topology"
    if signature == "yamada:contractible-vertex":
        return "vertex"
    if signature == "error:core" or signature.startswith("error:core"):
        return "core rejected"
    if signature == "error:projection" or "projection" in signature or "overlapping" in signature or "Nongeneric" in signature:
        return "projection"
    if signature.startswith("error:"):
        return "error"
    if signature.startswith("overview:"):
        return signature[len("overview:"):].replace("-", " ")
    if signature.startswith("yamada:"):
        return yamada_labels.get(signature, "Y")
    if signature.startswith("abstract-yamada:"):
        return yamada_labels.get(signature, "G")
    if signature.startswith("graph:"):
        try:
            nodes, edges, components, cycle_rank, _ = ast.literal_eval(signature[len("graph:"):])
            return f"n{nodes}/e{edges}/c{components}/r{cycle_rank}"
        except Exception:
            return "graph"
    return signature[:16]


def compact_transition_record(transition, yamada_labels=None):
    row = dict(transition)
    for key in ("phase_left", "phase_right"):
        if key in row:
            row[key] = compact_phase_label(row[key], yamada_labels)
    return row


def print_transition_summary(transitions, *, yamada_labels=None, heading="Transition intervals", limit=40):
    print(f"\n{heading}: {len(transitions)}")
    for transition in transitions[:limit]:
        print(compact_transition_record(transition, yamada_labels))
    if len(transitions) > limit:
        print(f"... {len(transitions) - limit} more intervals omitted")


def compact_legend_row(row):
    signature = row.get("signature", "")
    compact = {
        key: row[key]
        for key in ("phase_id", "phase")
        if key in row
    }
    if signature.startswith("abstract-yamada:"):
        compact["source"] = "abstract graph Yamada"
    elif signature.startswith("yamada:"):
        compact["source"] = "embedded Yamada"
    elif signature:
        compact["signature"] = signature
    if "yamada_polynomial" in row:
        compact["yamada_polynomial"] = row["yamada_polynomial"]
    return compact


def print_legend_rows(rows, *, limit=20):
    for row in rows[:limit]:
        print(compact_legend_row(row))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more legend rows omitted")


def phase_sort_key(signature):
    if signature.startswith("topology:"):
        order = {
            "topology:boundary": 0,
            "topology:open": 1,
            "topology:noncompact-pole": 2,
            "topology:fragmented": 3,
            "topology:multi-component": 3,
            "topology:tube": 4,
            "topology:sphere": 5,
        }
        if signature in order:
            return (order[signature], signature)
        if signature.startswith("topology:higher-genus"):
            return (6, signature)
        if signature.startswith("topology:closed"):
            return (7, signature)
        return (8, signature)
    if signature == "error:boundary":
        return (0, signature)
    if signature == "error:chart-gap":
        return (1, signature)
    if signature.startswith("error:topology"):
        return (2, signature)
    if signature.startswith("error:core"):
        return (3, signature)
    if signature.startswith("error:projection"):
        return (4, signature)
    if signature.startswith("error:"):
        return (5, signature)
    if signature.startswith("overview:"):
        order = {
            "overview:boundary": 0,
            "overview:chart-gap": 1,
            "overview:projection": 2,
            "overview:embedded-yamada": 3,
            "overview:graph-yamada": 4,
            "overview:vertex": 5,
        }
        return (order.get(signature, 6), signature)
    if signature.startswith("yamada:"):
        return (10, signature)
    if signature.startswith("abstract-yamada:"):
        return (11, signature)
    return (20, signature)


def ordered_phase_grid(result):
    signatures = sorted({record.phase_signature for record in result.records}, key=phase_sort_key)
    ids = {signature: index for index, signature in enumerate(signatures)}
    grid = np.empty((len(result.radii), len(result.lambdas)), dtype=int)
    lookup = {(record.lam, record.radius): record for record in result.records}
    for row, radius in enumerate(result.radii):
        for column, lam in enumerate(result.lambdas):
            grid[row, column] = ids[lookup[(float(lam), float(radius))].phase_signature]
    return grid, {index: signature for signature, index in ids.items()}


def sample_edges(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 1:
        delta = 0.5
        return np.array([values[0] - delta, values[0] + delta])
    mids = 0.5 * (values[:-1] + values[1:])
    first = values[0] - (mids[0] - values[0])
    last = values[-1] + (values[-1] - mids[-1])
    return np.concatenate([[first], mids, [last]])


def dynamic_phase_colors(signatures, cmap_name, *, start=0.06, stop=0.94):
    if not signatures:
        return {}
    cmap = plt.get_cmap(cmap_name)
    if len(signatures) == 1:
        values = [0.5]
    else:
        values = np.linspace(start, stop, len(signatures))
    return {
        signature: mcolors.to_hex(cmap(float(value)))
        for signature, value in zip(signatures, values)
    }


def phase_colormap(legend, compact_labels):
    ordered_signatures = [legend[idx] for idx in sorted(legend)]
    embedded_signatures = [
        signature for signature in ordered_signatures
        if signature.startswith("yamada:") and signature != "yamada:contractible-vertex"
    ]
    abstract_signatures = [
        signature for signature in ordered_signatures
        if signature.startswith("abstract-yamada:")
    ]
    dynamic_colors = {}
    dynamic_colors.update(dynamic_phase_colors(embedded_signatures, "viridis", start=0.12, stop=0.82))
    dynamic_colors.update(dynamic_phase_colors(abstract_signatures, "turbo", start=0.04, stop=0.96))
    colors = []
    for idx in sorted(legend):
        signature = legend[idx]
        if signature.startswith("topology:"):
            if signature == "topology:boundary" or signature == "topology:open":
                colors.append("#d8d8d8")
            elif signature == "topology:noncompact-pole":
                colors.append("#6b7280")
            elif signature == "topology:fragmented" or signature == "topology:multi-component":
                colors.append("#f59e0b")
            elif signature == "topology:tube":
                colors.append("#2563eb")
            elif signature == "topology:sphere":
                colors.append("#16a34a")
            elif signature.startswith("topology:higher-genus"):
                colors.append("#9333ea")
            else:
                colors.append("#0f766e")
        elif signature == "error:boundary":
            colors.append("#d8d8d8")
        elif signature == "error:chart-gap":
            colors.append("#9ca3af")
        elif signature.startswith("error:topology"):
            colors.append("#7a7a7a")
        elif signature == "yamada:contractible-vertex":
            colors.append("#0f766e")
        elif signature.startswith("error:core"):
            colors.append("#a16207")
        elif signature.startswith("error:projection"):
            colors.append("#f97316")
        elif signature.startswith("error:"):
            colors.append("#111827")
        elif signature.startswith("overview:"):
            overview_colors = {
                "overview:boundary": "#d8d8d8",
                "overview:chart-gap": "#9ca3af",
                "overview:projection": "#f97316",
                "overview:embedded-yamada": "#2563eb",
                "overview:graph-yamada": "#a855f7",
                "overview:vertex": "#16a34a",
            }
            colors.append(overview_colors.get(signature, "#64748b"))
        elif signature.startswith("yamada:") or signature.startswith("abstract-yamada:"):
            colors.append(dynamic_colors.get(signature, "#64748b"))
        else:
            colors.append("#64748b")
    return mcolors.ListedColormap(colors)


def select_phase_colorbar_ticks(ticks, compact_labels, legend, *, max_labels=14):
    tick_to_label = dict(zip(ticks, compact_labels))
    if len(ticks) <= max_labels:
        selected = list(ticks)
    else:
        positions = np.linspace(0, len(ticks) - 1, max_labels)
        selected = [ticks[int(round(position))] for position in positions]
        selected = sorted(dict.fromkeys(selected))
    return selected, [f"P{idx} {tick_to_label[idx]}" for idx in selected]


def plot_phase_signature_grid(result, *, title="Yamada phase signatures", ax=None):
    labels, legend = ordered_phase_grid(result)
    if ax is None:
        _, ax = plt.subplots(figsize=(9.8, 5.8))
    yamada_labels = phase_label_lookup(result)
    yamada_polynomials = phase_polynomial_lookup(result)
    ticks = sorted(legend)
    compact_labels = [compact_phase_label(legend[idx], yamada_labels) for idx in ticks]
    cmap = phase_colormap(legend, compact_labels)
    norm = mcolors.BoundaryNorm(np.arange(-0.5, len(ticks) + 0.5, 1), cmap.N)
    cell_count = labels.size
    edge_alpha = 0.22 if cell_count <= 600 else 0.10
    line_width = 0.18 if cell_count <= 600 else 0.05
    image = ax.pcolormesh(
        sample_edges(result.lambdas), sample_edges(result.radii), labels,
        cmap=cmap, norm=norm, shading="flat",
        edgecolors=(1, 1, 1, edge_alpha), linewidth=line_width,
    )
    ax.set_xlim(float(result.lambdas[0]), float(result.lambdas[-1]))
    ax.set_ylim(float(result.radii[0]), float(result.radii[-1]))
    ax.set_xlabel(r"$\lambda$")
    ax.set_ylabel(r"level radius $\epsilon$")
    ax.set_title(title)
    cbar_ticks, cbar_ticklabels = select_phase_colorbar_ticks(ticks, compact_labels, legend)
    cbar = plt.colorbar(image, ax=ax, shrink=0.84, ticks=cbar_ticks)
    cbar.ax.set_yticklabels(cbar_ticklabels, fontsize=8)
    legend_rows = []
    for idx, label in zip(ticks, compact_labels):
        signature = legend[idx]
        row = {"phase_id": f"P{idx}", "phase": label, "signature": signature}
        if signature in yamada_polynomials:
            row["yamada_polynomial"] = yamada_polynomials[signature]
        legend_rows.append(row)
    image._knotted_graph_phase_legend_rows = legend_rows
    return ax, legend_rows


def plot_phase_observable(result, observable, *, title=None, ax=None):
    grid = result.record_grid()
    values = np.empty(grid.shape, dtype=float)
    for row_index, records in enumerate(grid):
        for column_index, record in enumerate(records):
            values[row_index, column_index] = np.nan if record.error else float(getattr(record, observable))
    if ax is None:
        _, ax = plt.subplots()
    image = ax.pcolormesh(
        sample_edges(result.lambdas), sample_edges(result.radii), values,
        shading="flat", edgecolors=(1, 1, 1, 0.18), linewidth=0.12,
    )
    ax.set_xlim(float(result.lambdas[0]), float(result.lambdas[-1]))
    ax.set_ylim(float(result.radii[0]), float(result.radii[-1]))
    ax.set_xlabel(r"$\lambda$")
    ax.set_ylabel(r"level radius $\epsilon$")
    ax.set_title(title or observable.replace("_", " "))
    plt.colorbar(image, ax=ax, shrink=0.82)
    return ax


def normalize_scan_error(error):
    if error is None:
        return None
    if "sampling-box boundary" in error:
        return "boundary"
    if "projection pole" in error or "S3 vertex bound" in error:
        return "chart-gap"
    if "topology" in error or "non-tubular" in error:
        return "topology"
    if "core" in error or "leaf" in error or "no edges" in error:
        return "core"
    if (
        "projection" in error or "overlapping" in error
        or "Nongeneric" in error or "colinear" in error or "tangent" in error
    ):
        return "projection"
    return "numerical"


def normalize_scan_error_phases(result):
    records = []
    for record in result.records:
        signature = record.phase_signature
        category = normalize_scan_error(record.error)
        if category is not None:
            if category == "topology" and record.error:
                signature = "error:topology"
            else:
                signature = f"error:{category}"
        if signature != record.phase_signature:
            record = dataclasses.replace(record, phase_signature=signature)
        records.append(record)
    return dataclasses.replace(result, records=records)


def diagnostic_is_certified_knot_tube(diagnostic):
    if diagnostic.touches_box_boundary or not diagnostic.surface_is_closed:
        return False
    if diagnostic.matches_expected_tubular_neighborhood is False:
        return False
    if diagnostic.matches_expected_tubular_neighborhood is True:
        return True
    return (
        diagnostic.volume_components == 1
        and diagnostic.surface_components == 1
        and diagnostic.total_boundary_genus == 1
    )


def topology_error_message(diagnostic):
    if diagnostic.touches_box_boundary:
        return "boundary: sublevel set touches the sampling-box boundary"
    return (
        "topology: non-tubular level set "
        f"V={diagnostic.volume_components}, S={diagnostic.surface_components}, "
        f"closed={diagnostic.surface_is_closed}, genus={diagnostic.total_boundary_genus}"
    )


def graph_signature_values(graph):
    components = nx.number_connected_components(graph) if graph.number_of_nodes() else 0
    cycle_rank = graph.number_of_edges() - graph.number_of_nodes() + components
    degree_sequence = tuple(sorted((degree for _, degree in graph.degree()), reverse=True))
    return graph.number_of_nodes(), graph.number_of_edges(), components, cycle_rank, degree_sequence


def prepare_yamada_core_graph(graph, *, smooth_eps=None):
    smooth_eps = YAMADA_GRAPH_SMOOTH_EPS if smooth_eps is None else float(smooth_eps)
    core = remove_leaf_nodes(graph)
    if core.number_of_edges() == 0:
        return core
    core = simplify_edges(core)
    if core.number_of_edges() == 0:
        return core
    smoothing_attempts = [smooth_eps]
    if smooth_eps != YAMADA_LINK_CORE_SMOOTH_EPS:
        smoothing_attempts.append(YAMADA_LINK_CORE_SMOOTH_EPS)
    last_exc = None
    for eps in smoothing_attempts:
        try:
            return smooth_edges(core, epsilon=eps, copy=True)
        except ValueError as exc:
            last_exc = exc
            if "collapsed to fewer than two distinct points" not in str(exc):
                raise
    # Tiny sampled loops can be topologically valid while unsafe to smooth.
    # Keep the simplified unsmoothed core rather than rejecting the phase cell.
    return core


def is_closed_link_core_signature(components, cycle_rank, degree_sequence):
    return cycle_rank == components and all(degree == 2 for degree in degree_sequence)


def validate_yamada_core_graph(graph):
    nodes, edges, components, cycle_rank, degree_sequence = graph_signature_values(graph)
    if edges == 0 or components == 0:
        raise ValueError(
            "core: extracted Yamada core has no closed edges "
            f"nodes={nodes}, edges={edges}, degrees={degree_sequence}"
        )
    if any(degree == 1 for degree in degree_sequence):
        raise ValueError(
            "core: leaf artifacts remain after pruning "
            f"nodes={nodes}, edges={edges}, degrees={degree_sequence}"
        )
    if YAMADA_REQUIRE_LINK_CORE:
        if cycle_rank != components or any(degree != 2 for degree in degree_sequence):
            raise ValueError(
                "core: expected a closed knot/link core for this phase map "
                f"nodes={nodes}, edges={edges}, components={components}, "
                f"cycle_rank={cycle_rank}, degrees={degree_sequence}"
            )
    elif cycle_rank <= 0:
        raise ValueError(
            "core: extracted graph has no cycle structure "
            f"nodes={nodes}, edges={edges}, components={components}, "
            f"cycle_rank={cycle_rank}, degrees={degree_sequence}"
        )
    return nodes, edges, components, cycle_rank, degree_sequence


def nearest_scan_record(result, lam, radius):
    if result is None:
        return None
    lambdas = np.asarray(result.lambdas, dtype=float)
    radii = np.asarray(result.radii, dtype=float)
    if len(lambdas) == 0 or len(radii) == 0:
        return None
    lam_index = int(np.argmin(np.abs(lambdas - float(lam))))
    radius_index = int(np.argmin(np.abs(radii - float(radius))))

    def grid_tolerance(values):
        if len(values) < 2:
            return np.inf
        diffs = np.diff(np.sort(values))
        return 0.51 * float(np.min(diffs))

    if abs(float(lambdas[lam_index]) - float(lam)) > grid_tolerance(lambdas):
        return None
    if abs(float(radii[radius_index]) - float(radius)) > grid_tolerance(radii):
        return None
    target = (float(lambdas[lam_index]), float(radii[radius_index]))
    for record in result.records:
        if np.isclose(record.lam, target[0]) and np.isclose(record.radius, target[1]):
            return record
    return None


def semiholomorphic_s3_abs_upper_bound(field):
    polynomial = getattr(field, "semiholomorphic", None)
    if polynomial is None:
        return None
    return float(sum(abs(coefficient) for coefficient in polynomial.terms.values()))


def certified_full_s3_vertex_phase(field, radius):
    bound = semiholomorphic_s3_abs_upper_bound(field)
    if bound is None:
        return None
    threshold = bound * (1.0 + S3_VERTEX_BOUND_MARGIN)
    if float(radius) >= threshold:
        return {"s3_abs_upper_bound": bound, "threshold": threshold}
    return None


def topology_certifies_vertex_phase(lam, radius, *, field=None):
    if field is not None and certified_full_s3_vertex_phase(field, radius) is not None:
        return True
    topology_result = globals().get("topology_scan_result")
    record = nearest_scan_record(topology_result, lam, radius)
    return record is not None and record.phase_signature == "topology:sphere"


def projection_retry_samples_for_options(yamada_options):
    base_count = int(yamada_options.get("num_rotation_samples", YAMADA_NUM_ROTATION_SAMPLES))
    explicit = yamada_options.get("projection_retry_samples")
    if explicit is not None:
        samples = [int(value) for value in explicit]
    elif base_count >= 64:
        samples = [base_count, min(2 * base_count, 256), min(4 * base_count, 512)]
    else:
        samples = [base_count]
    return tuple(dict.fromkeys(value for value in samples if value > 0))


def compute_embedded_yamada_polynomial_with_retries(
    compute_yamada_polynomial,
    graph,
    variable,
    yamada_options,
    projection_retry_samples,
):
    base_options = dict(yamada_options)
    base_options.pop("projection_retry_samples", None)
    last_exc = None
    for sample_count in projection_retry_samples:
        options = dict(base_options)
        options["num_rotation_samples"] = int(sample_count)
        try:
            return compute_yamada_polynomial(graph, variable, **options)
        except Exception as exc:
            last_exc = exc
            category = normalize_scan_error(f"{type(exc).__name__}: {exc}")
            if category != "projection":
                raise
    raise last_exc


def run_certified_yamada_scan(path, *, lambdas, radii, dimension, yamada_options=None, graph_options=None):
    from knotted_graph.invariants.yamada import compute_graph_yamada_polynomial
    from knotted_graph.projection import compute_yamada_polynomial
    yamada_options = dict(yamada_options or {})
    graph_options = dict(graph_options or {})
    projection_retry_samples = projection_retry_samples_for_options(yamada_options)
    variable = sp.Symbol("A")
    records = []
    for lam in lambdas:
        field = path.at(float(lam))
        sample = field.sample(span=SPAN, dimension=dimension)
        for radius in radii:
            graph = nx.MultiGraph()
            yamada = None
            error = None
            try:
                if topology_certifies_vertex_phase(float(lam), float(radius), field=field):
                    vertex_graph = nx.MultiGraph()
                    vertex_graph.add_node(0)
                    graph = vertex_graph
                    yamada = compute_graph_yamada_polynomial(vertex_graph, variable)
                else:
                    pole = field.projection_pole_value
                    if pole is not None and float(radius) >= abs(pole):
                        bound = semiholomorphic_s3_abs_upper_bound(field)
                        if bound is None:
                            raise ValueError(
                                "projection pole is inside this R3 chart and no semiholomorphic S3 bound is available"
                            )
                        raise ValueError(
                            "projection pole is inside this R3 chart before the certified S3 vertex bound "
                            f"epsilon >= {bound * (1.0 + S3_VERTEX_BOUND_MARGIN):.6g}"
                        )
                    raw_graph = field.to_spatial_graph(
                        float(radius), sample=sample, span=SPAN,
                        dimension=dimension, **graph_options,
                    )
                    graph = prepare_yamada_core_graph(raw_graph)
                    if graph.number_of_edges() == 0:
                        vertex_graph = nx.MultiGraph()
                        vertex_graph.add_node(0)
                        yamada = compute_graph_yamada_polynomial(vertex_graph, variable)
                        graph = vertex_graph
                    else:
                        nodes, edges, components, cycle_rank, degree_sequence = validate_yamada_core_graph(graph)
                        use_embedded_yamada = (
                            is_closed_link_core_signature(components, cycle_rank, degree_sequence)
                            or edges <= YAMADA_EMBEDDED_BRANCH_EDGE_LIMIT
                        )
                        if use_embedded_yamada:
                            try:
                                yamada = compute_embedded_yamada_polynomial_with_retries(
                                    compute_yamada_polynomial,
                                    graph,
                                    variable,
                                    yamada_options,
                                    projection_retry_samples,
                                )
                            except Exception as exc:
                                category = normalize_scan_error(f"{type(exc).__name__}: {exc}")
                                if category != "projection" or not is_closed_link_core_signature(components, cycle_rank, degree_sequence):
                                    raise
                                graph = prepare_yamada_core_graph(raw_graph, smooth_eps=YAMADA_LINK_CORE_SMOOTH_EPS)
                                nodes, edges, components, cycle_rank, degree_sequence = validate_yamada_core_graph(graph)
                                if not is_closed_link_core_signature(components, cycle_rank, degree_sequence):
                                    raise
                                yamada = compute_embedded_yamada_polynomial_with_retries(
                                    compute_yamada_polynomial,
                                    graph,
                                    variable,
                                    yamada_options,
                                    projection_retry_samples,
                                )
                        else:
                            yamada = compute_graph_yamada_polynomial(graph, variable)
            except Exception as exc:
                error = f"{type(exc).__name__}: {exc}"
            nodes, edges, components, cycle_rank, degree_sequence = graph_signature_values(graph)
            if yamada is not None:
                if graph.number_of_nodes() == 1 and graph.number_of_edges() == 0:
                    phase_signature = "yamada:contractible-vertex"
                elif graph.number_of_edges() > YAMADA_EMBEDDED_BRANCH_EDGE_LIMIT and not is_closed_link_core_signature(components, cycle_rank, degree_sequence):
                    phase_signature = "abstract-yamada:" + sp.srepr(sp.expand(yamada))
                else:
                    phase_signature = "yamada:" + sp.srepr(sp.expand(yamada))
            else:
                category = normalize_scan_error(error)
                phase_signature = f"error:{category or 'numerical'}"
            records.append(KnotDeformationRecord(
                lam=float(lam), radius=float(radius), nodes=nodes, edges=edges,
                components=components, cycle_rank=cycle_rank,
                degree_sequence=degree_sequence, yamada=yamada,
                phase_signature=phase_signature, error=error,
            ))
    return KnotDeformationScanResult(
        lambdas=np.asarray(lambdas, dtype=float),
        radii=np.asarray(radii, dtype=float),
        records=records,
    )


def overview_phase_signature(signature):
    if signature == "yamada:contractible-vertex":
        return "overview:vertex"
    if signature.startswith("yamada:"):
        return "overview:embedded-yamada"
    if signature.startswith("abstract-yamada:"):
        return "overview:graph-yamada"
    if signature == "error:boundary" or signature == "topology:boundary":
        return "overview:boundary"
    if signature == "error:chart-gap" or signature == "topology:noncompact-pole":
        return "overview:chart-gap"
    if signature.startswith("error:projection"):
        return "overview:projection"
    if signature.startswith("error:"):
        return "overview:" + compact_phase_label(signature).replace(" ", "-")
    return signature


def phase_overview_result(result):
    return dataclasses.replace(
        result,
        records=[
            dataclasses.replace(record, phase_signature=overview_phase_signature(record.phase_signature))
            for record in result.records
        ],
    )


def compactified_radius_grid_for_path(
    path,
    lambdas,
    *,
    base_radii=None,
    high_count=26,
    safety=1.05,
):
    base_radii = np.asarray(GALLERY_PHASE_RADII if base_radii is None else base_radii, dtype=float)
    s3_bounds = np.asarray([
        semiholomorphic_s3_abs_upper_bound(path.at(float(lam)))
        for lam in lambdas
    ], dtype=float)
    if not np.all(np.isfinite(s3_bounds)):
        raise AssertionError("compactified scan requires finite semiholomorphic S3 bounds")
    certified_top = float(np.max(s3_bounds) * (1.0 + S3_VERTEX_BOUND_MARGIN))
    compact_radius_max = certified_top * float(safety)
    high_start = max(float(base_radii[-1]) * 1.15, 0.42)
    if compact_radius_max <= high_start:
        compact_high_radii = np.asarray([compact_radius_max], dtype=float)
    else:
        compact_high_radii = np.geomspace(high_start, compact_radius_max, int(high_count))
    compact_phase_radii = np.unique(np.round(np.concatenate([base_radii, compact_high_radii]), 10))
    if float(compact_phase_radii[-1]) < certified_top:
        compact_phase_radii = np.unique(np.round(np.concatenate([compact_phase_radii, [certified_top * safety]]), 10))
    return compact_phase_radii, s3_bounds


def compactified_yamada_error_counts(result):
    return Counter(
        normalize_scan_error(record.error) if record.error else "ok"
        for record in result.records
    )


def audit_compactified_yamada_result(result, *, allowed_nonfatal=("boundary", "chart-gap")):
    error_counts = compactified_yamada_error_counts(result)
    bad_error_counts = Counter({
        key: value
        for key, value in error_counts.items()
        if key != "ok" and key not in allowed_nonfatal and value
    })
    top_radius = float(np.max(result.radii))
    top_records = [record for record in result.records if np.isclose(record.radius, top_radius)]
    top_phase_counts = Counter(record.phase_signature for record in top_records)
    if set(top_phase_counts) != {"yamada:contractible-vertex"}:
        raise AssertionError(
            "highest radius is not certified as the all-S3 / vertex Yamada phase: "
            f"{top_phase_counts}"
        )
    if bad_error_counts:
        raise AssertionError(
            "unexpected computational errors remain in compactified Yamada scan: "
            f"{bad_error_counts}"
        )
    return {
        "records": len(result.records),
        "valid_yamada": sum(record.yamada is not None for record in result.records),
        "error_counts": error_counts,
        "bad_error_counts": bad_error_counts,
        "top_radius": top_radius,
        "top_phase_counts": top_phase_counts,
        "vertex_records": sum(
            record.phase_signature == "yamada:contractible-vertex"
            for record in result.records
        ),
    }


def run_compactified_yamada_transition(
    label,
    path,
    *,
    lambdas=None,
    base_radii=None,
    dimension=None,
    yamada_options=None,
    high_count=26,
):
    lambdas = np.asarray(GALLERY_LAMBDAS if lambdas is None else lambdas, dtype=float)
    dimension = GALLERY_PHASE_DIM if dimension is None else int(dimension)
    radii, s3_bounds = compactified_radius_grid_for_path(
        path, lambdas, base_radii=base_radii, high_count=high_count
    )
    print(f"{label}: S3 |F| certified upper-bound range:", (float(np.min(s3_bounds)), float(np.max(s3_bounds))))
    print(f"{label}: compactified radii:", (float(radii[0]), float(radii[-1]), len(radii)))
    result = cached_deformation_scan(
        f"{label}_yamada_compactified_certified",
        path,
        lambdas=lambdas,
        radii=radii,
        dimension=dimension,
        invariant="yamada",
        yamada_options=yamada_options or YAMADA_OPTIONS,
    )
    audit = audit_compactified_yamada_result(result)
    print(f"{label}: compactified audit:", {
        "records": audit["records"],
        "valid_yamada": audit["valid_yamada"],
        "errors": dict(audit["error_counts"]),
        "top_radius": audit["top_radius"],
        "top_phase_counts": dict(audit["top_phase_counts"]),
    })
    return result, phase_overview_result(result), radii, s3_bounds, audit


def plot_compactified_overview_gallery(gallery_results, *, title):
    ordered = [
        "overview:boundary",
        "overview:chart-gap",
        "overview:projection",
        "overview:embedded-yamada",
        "overview:graph-yamada",
        "overview:vertex",
    ]
    present = [
        signature
        for signature in ordered
        if any(
            any(record.phase_signature == signature for record in bundle["overview"].records)
            for bundle in gallery_results
        )
    ]
    ids = {signature: index for index, signature in enumerate(present)}
    colors = {
        "overview:boundary": "#d8d8d8",
        "overview:chart-gap": "#9ca3af",
        "overview:projection": "#f97316",
        "overview:embedded-yamada": "#2563eb",
        "overview:graph-yamada": "#a855f7",
        "overview:vertex": "#16a34a",
    }
    cmap = mcolors.ListedColormap([colors[signature] for signature in present])
    norm = mcolors.BoundaryNorm(np.arange(-0.5, len(present) + 0.5, 1), cmap.N)
    fig, axes = plt.subplots(
        len(gallery_results), 1,
        figsize=(10.5, 2.35 * len(gallery_results)),
        constrained_layout=True,
        sharex=True,
    )
    axes = np.atleast_1d(axes)
    last_image = None
    for ax, bundle in zip(axes, gallery_results):
        result = bundle["overview"]
        grid = np.empty((len(result.radii), len(result.lambdas)), dtype=int)
        lookup = {(record.lam, record.radius): record for record in result.records}
        for row, radius in enumerate(result.radii):
            for column, lam in enumerate(result.lambdas):
                grid[row, column] = ids[lookup[(float(lam), float(radius))].phase_signature]
        cell_count = grid.size
        last_image = ax.pcolormesh(
            sample_edges(result.lambdas),
            sample_edges(result.radii),
            grid,
            cmap=cmap,
            norm=norm,
            shading="flat",
            edgecolors=(1, 1, 1, 0.08 if cell_count > 600 else 0.18),
            linewidth=0.04 if cell_count > 600 else 0.12,
        )
        ax.set_yscale("log")
        ax.set_xlim(float(result.lambdas[0]), float(result.lambdas[-1]))
        ax.set_ylim(float(result.radii[0]), float(result.radii[-1]))
        ax.set_ylabel(r"$\epsilon$")
        ax.set_title(bundle["title"], loc="left", fontsize=11)
    axes[-1].set_xlabel(r"$\lambda$")
    fig.suptitle(title, fontsize=16)
    if last_image is not None:
        cbar = fig.colorbar(last_image, ax=axes, shrink=0.92, ticks=range(len(present)))
        cbar.ax.set_yticklabels([compact_phase_label(signature) for signature in present])
    return fig, axes


def cached_deformation_scan(label, path, *, lambdas, radii, dimension, invariant=None, yamada_options=None, graph_options=None):
    yamada_options = dict(yamada_options or {})
    graph_options = dict(graph_options or {})
    payload = {
        "version": PHASE_SCAN_CACHE_VERSION,
        "label": label,
        "start": getattr(path.start, "name", "start"),
        "end": getattr(path.end, "name", "end"),
        "lambdas": [round(float(value), 10) for value in lambdas],
        "radii": [round(float(value), 10) for value in radii],
        "dimension": dimension,
        "invariant": invariant,
        "yamada_options": yamada_options,
        "graph_options": graph_options,
        "validate_yamada_tubes": VALIDATE_YAMADA_TUBES,
        "yamada_graph_smooth_eps": YAMADA_GRAPH_SMOOTH_EPS,
        "yamada_require_link_core": YAMADA_REQUIRE_LINK_CORE,
        "s3_vertex_bound_margin": S3_VERTEX_BOUND_MARGIN,
    }
    digest = hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]
    cache_path = PHASE_CACHE_DIR / f"{label}_{digest}.pkl"
    if cache_path.exists():
        with cache_path.open("rb") as handle:
            result = pickle.load(handle)
        result = normalize_scan_error_phases(result)
        cached_error_counts = compactified_yamada_error_counts(result)
        high_resolution_retry_cache = (
            invariant == "yamada"
            and int(yamada_options.get("num_rotation_samples", 0)) >= 64
            and (cached_error_counts.get("projection", 0) or cached_error_counts.get("numerical", 0))
        )
        if high_resolution_retry_cache:
            print(
                f"discarding retryable-error cache {cache_path}: "
                f"projection={cached_error_counts.get('projection', 0)}, "
                f"numerical={cached_error_counts.get('numerical', 0)}"
            )
        else:
            print(f"loaded cached scan: {cache_path}")
            return result

    t0 = time.perf_counter()
    if invariant == "yamada" and VALIDATE_YAMADA_TUBES:
        result = run_certified_yamada_scan(
            path,
            lambdas=lambdas,
            radii=radii,
            dimension=dimension,
            yamada_options=yamada_options,
            graph_options=graph_options,
        )
    else:
        scan = KnotDeformationScan(
            path,
            lambdas=lambdas,
            radii=radii,
            span=SPAN,
            dimension=dimension,
            invariant=invariant,
            yamada_options=yamada_options,
            graph_options=graph_options,
            continue_on_error=True,
        )
        result = scan.run()
    with cache_path.open("wb") as handle:
        pickle.dump(result, handle)
    elapsed = time.perf_counter() - t0
    print(f"computed scan: {len(result.records)} records in {elapsed:.2f}s")
    print(f"cached scan: {cache_path}")
    return normalize_scan_error_phases(result)


In [ ]:

@dataclasses.dataclass(frozen=True)
class LevelSetTopologyRecord:
    lam: float
    radius: float
    occupied_voxels: int | None
    volume_components: int | None
    surface_components: int | None
    surface_euler_characteristic: int | None
    surface_is_closed: bool | None
    touches_box_boundary: bool | None
    total_boundary_genus: int | None
    expected_components: int | None
    matches_expected_tubular_neighborhood: bool | None
    phase_signature: str
    error: str | None = None


@dataclasses.dataclass(frozen=True)
class LevelSetTopologyScanResult:
    lambdas: np.ndarray
    radii: np.ndarray
    records: list[LevelSetTopologyRecord]

    def record_grid(self):
        lookup = {(record.lam, record.radius): record for record in self.records}
        return np.asarray([
            [lookup[(float(lam), float(radius))] for lam in self.lambdas]
            for radius in self.radii
        ], dtype=object)

    def transition_points(self):
        transitions = []
        grid = self.record_grid()
        for row, radius in enumerate(self.radii):
            row_records = grid[row]
            for left, right in zip(row_records[:-1], row_records[1:]):
                if left.phase_signature != right.phase_signature:
                    transitions.append({
                        "radius": float(radius),
                        "lambda_left": float(left.lam),
                        "lambda_right": float(right.lam),
                        "phase_left": left.phase_signature,
                        "phase_right": right.phase_signature,
                    })
        return transitions


def topology_phase_signature(diagnostic):
    if diagnostic.touches_box_boundary:
        return "topology:boundary"
    if not diagnostic.surface_is_closed:
        return "topology:open"
    if diagnostic.total_boundary_genus is None:
        return "topology:open"
    if diagnostic.volume_components != 1 or diagnostic.surface_components != 1:
        return "topology:multi-component"
    if diagnostic.total_boundary_genus == 0:
        return "topology:sphere"
    if diagnostic.total_boundary_genus == 1:
        return "topology:tube"
    return "topology:higher-genus"


def topology_error_signature(error):
    if error and "projection pole" in error:
        return "topology:noncompact-pole"
    if error and "sampling-box boundary" in error:
        return "topology:boundary"
    return "topology:open"


def normalize_topology_scan_phases(result):
    records = []
    for record in result.records:
        if record.error:
            phase_signature = topology_error_signature(record.error)
        else:
            phase_signature = topology_phase_signature(record)
        records.append(dataclasses.replace(record, phase_signature=phase_signature))
    return dataclasses.replace(result, records=records)


def run_levelset_topology_scan(path, *, lambdas, radii, dimension, span):
    records = []
    for lam in lambdas:
        field = path.at(float(lam))
        sample = field.sample(span=span, dimension=dimension)
        for radius in radii:
            error = None
            diagnostic = None
            try:
                diagnostic = field.diagnose_level(
                    float(radius), sample=sample, span=span, dimension=dimension
                )
                phase_signature = topology_phase_signature(diagnostic)
            except Exception as exc:
                error = f"{type(exc).__name__}: {exc}"
                phase_signature = topology_error_signature(error)
            records.append(LevelSetTopologyRecord(
                lam=float(lam),
                radius=float(radius),
                occupied_voxels=None if diagnostic is None else diagnostic.occupied_voxels,
                volume_components=None if diagnostic is None else diagnostic.volume_components,
                surface_components=None if diagnostic is None else diagnostic.surface_components,
                surface_euler_characteristic=None if diagnostic is None else diagnostic.surface_euler_characteristic,
                surface_is_closed=None if diagnostic is None else diagnostic.surface_is_closed,
                touches_box_boundary=None if diagnostic is None else diagnostic.touches_box_boundary,
                total_boundary_genus=None if diagnostic is None else diagnostic.total_boundary_genus,
                expected_components=None if diagnostic is None else diagnostic.expected_components,
                matches_expected_tubular_neighborhood=None if diagnostic is None else diagnostic.matches_expected_tubular_neighborhood,
                phase_signature=phase_signature,
                error=error,
            ))
    return LevelSetTopologyScanResult(
        lambdas=np.asarray(lambdas, dtype=float),
        radii=np.asarray(radii, dtype=float),
        records=records,
    )


def topology_scan_to_cache_payload(result):
    return {
        "lambdas": np.asarray(result.lambdas, dtype=float).tolist(),
        "radii": np.asarray(result.radii, dtype=float).tolist(),
        "records": [dataclasses.asdict(record) for record in result.records],
    }


def topology_scan_from_cache_payload(payload):
    if isinstance(payload, LevelSetTopologyScanResult):
        return payload
    return LevelSetTopologyScanResult(
        lambdas=np.asarray(payload["lambdas"], dtype=float),
        radii=np.asarray(payload["radii"], dtype=float),
        records=[LevelSetTopologyRecord(**record) for record in payload["records"]],
    )


def cached_topology_phase_scan(label, path, *, lambdas, radii, dimension, span):
    payload = {
        "version": TOPOLOGY_SCAN_CACHE_VERSION,
        "label": label,
        "start": getattr(path.start, "name", "start"),
        "end": getattr(path.end, "name", "end"),
        "lambdas": [round(float(value), 10) for value in lambdas],
        "radii": [round(float(value), 10) for value in radii],
        "dimension": dimension,
        "span": span,
    }
    digest = hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]
    cache_path = PHASE_CACHE_DIR / f"{label}_{digest}.pkl"
    if cache_path.exists():
        try:
            with cache_path.open("rb") as handle:
                cached_result = topology_scan_from_cache_payload(pickle.load(handle))
            result = normalize_topology_scan_phases(cached_result)
            if any(
                before.phase_signature != after.phase_signature
                for before, after in zip(cached_result.records, result.records)
            ):
                with cache_path.open("wb") as handle:
                    pickle.dump(topology_scan_to_cache_payload(result), handle)
                print(f"normalized cached topology labels: {cache_path}")
            print(f"loaded cached topology scan: {cache_path}")
            return result
        except Exception as exc:
            print(f"discarding unreadable topology cache {cache_path}: {type(exc).__name__}: {exc}")
            cache_path.unlink(missing_ok=True)
    t0 = time.perf_counter()
    result = normalize_topology_scan_phases(run_levelset_topology_scan(
        path, lambdas=lambdas, radii=radii, dimension=dimension, span=span
    ))
    with cache_path.open("wb") as handle:
        pickle.dump(topology_scan_to_cache_payload(result), handle)
    elapsed = time.perf_counter() - t0
    print(f"computed topology scan: {len(result.records)} records in {elapsed:.2f}s")
    print(f"cached topology scan: {cache_path}")
    return result


def path_projection_pole_abs(path, lambdas):
    values = []
    for lam in lambdas:
        pole = path.at(float(lam)).projection_pole_value
        values.append(np.nan if pole is None else float(abs(pole)))
    return np.asarray(values, dtype=float)


def plot_projection_pole_threshold(ax, path, lambdas):
    pole_abs = path_projection_pole_abs(path, lambdas)
    if not np.isfinite(pole_abs).any():
        return pole_abs
    ymin, ymax = ax.get_ylim()
    visible = (pole_abs >= ymin) & (pole_abs <= ymax)
    if np.any(visible):
        ax.plot(lambdas, pole_abs, color="#111827", lw=1.15, ls="--", alpha=0.88)
        index = int(np.flatnonzero(visible)[0])
        ax.text(
            float(lambdas[index]), float(pole_abs[index]),
            " projection pole", ha="left", va="bottom",
            fontsize=7.5, color="#111827",
        )
    return pole_abs


def plot_topology_observable(result, observable, *, title=None, ax=None):
    grid = result.record_grid()
    values = np.empty(grid.shape, dtype=float)
    for row_index, records in enumerate(grid):
        for column_index, record in enumerate(records):
            value = getattr(record, observable)
            values[row_index, column_index] = np.nan if value is None else float(value)
    if ax is None:
        _, ax = plt.subplots()
    image = ax.pcolormesh(
        sample_edges(result.lambdas), sample_edges(result.radii), values,
        shading="flat", edgecolors=(1, 1, 1, 0.16), linewidth=0.10,
    )
    ax.set_xlim(float(result.lambdas[0]), float(result.lambdas[-1]))
    ax.set_ylim(float(result.radii[0]), float(result.radii[-1]))
    ax.set_xlabel(r"$\lambda$")
    ax.set_ylabel(r"level radius $\epsilon$")
    ax.set_title(title or observable.replace("_", " "))
    plt.colorbar(image, ax=ax, shrink=0.82)
    return ax


## 1. Built-in knot/link catalogue

The catalogue is deliberately small because arbitrary Artin braid words are supported.
Each entry stores a standard braid word, strand count, closure component count, aliases,
and a preferred construction where available.

In [ ]:
print("Available:", available_knot_names())

for name in available_knot_names():
    e = get_knot_entry(name)
    print({
        "name": e.canonical_name,
        "braid_word": e.braid_word,
        "strands": e.strands,
        "components": e.components,
        "torus_params": e.torus_params,
        "reference_field": e.reference_field,
        "aliases": e.aliases,
    })


## 2. Artin braid words: strands, permutation, and closure components

A braid word is encoded by integers:
- `+i` means \(\sigma_i\),
- `-i` means \(\sigma_i^{-1}\).

A standard figure-eight representative is


$$
\sigma_1\sigma_2^{-1}\sigma_1\sigma_2^{-1}.
$$



In [ ]:
word = (1, -2, 1, -2)

strands = infer_braid_strands(word)
perm = braid_permutation(word, strands)
components = braid_component_count(word, strands)

print("word:", word)
print("required strands for this word:", strands)
print("endpoint permutation:", perm)
print("components in closure:", components)


If the word contains \(\sigma_i\), that representation requires at least \(i+1\) strands.

This is not the global braid index


$$
b(K)=\min_{\widehat{\beta}=K} n(\beta),
$$


because the library is not searching over all Markov-equivalent braid representations.

## 3. Geometric braid representative

`geometric_braid_roots` returns one complex root per strand as the braid parameter
\(t\) runs from \(0\) to \(2\pi\).

In [ ]:
t = np.linspace(0, 2*np.pi, 600, endpoint=False)
roots = geometric_braid_roots(word, t, strands=strands)

fig = plt.figure(figsize=(8,5))
ax = fig.add_subplot(111, projection="3d")
for j in range(strands):
    ax.plot(t, roots[:,j].real, roots[:,j].imag)
ax.set_xlabel("t")
ax.set_ylabel("Re(root)")
ax.set_zlabel("Im(root)")
ax.set_title("Geometric Artin braid representative")
plt.show()


## 4. Braid \(\rightarrow\) semiholomorphic polynomial

The braid is encoded as roots of a monic polynomial in \(u\). Its periodic coefficient
functions are Fourier-expanded. Positive Fourier modes become powers of \(v\), and
negative modes become powers of \(\bar v\):


$$
f(u,v,\bar v)=\sum_{a,b,c}C_{abc}u^a v^b\bar v^c.
$$


The compiler increases Fourier bandwidth until the sampled root-tracking validation
criterion is satisfied.

In [ ]:
poly, report = braid_to_semiholomorphic(
    word,
    strands=strands,
    validation_samples=512 if QUICK else 2048,
)

print("degree_u:", poly.degree_u)
print("total_degree:", poly.total_degree)
print("retained terms:", len(poly.terms))
print("\nValidation report:")
for k, v in dataclasses.asdict(report).items():
    print(f"{k}: {v}")


In [ ]:
u, v, vbar = sp.symbols("u v vbar")
poly.to_sympy(u=u, v=v, vbar=vbar)


### What the braid validation means

The report includes the chosen Fourier cutoff, maximum sampled root error, minimum
target strand separation, their ratio, and `passed`.

This is a **numerical sampled braid-isotopy diagnostic** for the retained Fourier
approximation. It is not a formal proof of every analytic small-scaling condition on
\(S^3\).

## 5. Main user interface: `KnotFunction.from_braid`

Most users can skip the lower-level compiler and construct the field directly.

In [ ]:
braid_field = KnotFunction.from_braid(
    word,
    strands=strands,
    name="my_figure_eight_braid",
    validation_samples=512 if QUICK else 2048,
)

print("name:", braid_field.name)
print("expected components:", braid_field.expected_components)
print("construction:", braid_field.metadata["construction"])
print("braid word:", braid_field.metadata["braid_word"])
print("strands:", braid_field.metadata["strands"])
print("chart angle:", braid_field.s3_chart_angle)
print("validation passed:", braid_field.construction_report.passed)


The semiholomorphic field is defined on \(S^3\subset\mathbb C^2\) and composed with
inverse stereographic projection to obtain \(F:\mathbb R^3\to\mathbb C\).

In [ ]:
rng = np.random.default_rng(7)
xyz = rng.normal(size=(8,3))
uu, vv = inverse_stereographic_s3(xyz[:,0], xyz[:,1], xyz[:,2])

print("max | |u|^2+|v|^2-1 | =",
      np.max(np.abs(np.abs(uu)**2 + np.abs(vv)**2 - 1)))

print("\nR3 evaluations:")
print(braid_field(xyz[:,0], xyz[:,1], xyz[:,2]))

us, vs = sample_s3(6, seed=3)
print("\nS3 evaluations:")
print(braid_field.evaluate_s3(us, vs))
print("\nprojection pole value:", braid_field.projection_pole_value)


## 6. Symbolic \(F(x,y,z)\)
For semiholomorphic fields the stereographic substitution can be expanded into a SymPy expression in \(x,y,z\). Full rational cancellation is intentionally disabled in the quick path because it can dominate notebook runtime for braid-derived Fourier polynomials. The fast preview below shows the compact \((u,v,\bar v)\) polynomial and only computes the full \(F(x,y,z)\) expression when `RUN_FULL_SYMBOLIC_R3=True`.


In [ ]:
x, y, z = sp.symbols("x y z", real=True)
u_sym, v_sym, vbar_sym = sp.symbols("u v vbar")

if braid_field.semiholomorphic is None:
    expr_r3 = braid_field.symbolic_r3_expression((x, y, z))
    print("operation count:", sp.count_ops(expr_r3))
    display(expr_r3)
else:
    poly_s3_expr = braid_field.semiholomorphic.to_sympy(u=u_sym, v=v_sym, vbar=vbar_sym)
    print("Fast symbolic preview on S3 variables")
    print("terms:", len(braid_field.semiholomorphic.terms))
    print("degree_u:", braid_field.semiholomorphic.degree_u)
    print("total_degree:", braid_field.semiholomorphic.total_degree)
    print("operation count before R3 substitution:", sp.count_ops(poly_s3_expr))
    display(poly_s3_expr)

    if RUN_FULL_SYMBOLIC_R3:
        expr_r3 = braid_field.symbolic_r3_expression(
            (x, y, z), simplify=RUN_SYMBOLIC_SIMPLIFY
        )
        print("R3 operation count:", sp.count_ops(expr_r3))
        print("simplified with sp.cancel:", RUN_SYMBOLIC_SIMPLIFY)
        display(expr_r3)
    else:
        expr_r3 = None
        print(
            "Skipped full stereographic R3 expansion. "
            "Set RUN_FULL_SYMBOLIC_R3=True for final symbolic output; "
            "keep RUN_SYMBOLIC_SIMPLIFY=False for a faster unsimplified expression."
        )


## 7. Direct torus knots and links

`KnotFunction.torus(p,q)` constructs


$$
f(u,v)=u^p-v^q,
$$


with expected number of components \(\gcd(p,q)\).

In [ ]:
trefoil = KnotFunction.torus(2,3)
hopf = KnotFunction.torus(2,2)
cinquefoil = KnotFunction.torus(2,5)

for f in (trefoil, hopf, cinquefoil):
    print(f.name, "components =", f.expected_components, "metadata =", f.metadata)


## 8. Construct by standard knot/link name

`from_name` chooses the preferred built-in representation. Setting
`construction="braid"` forces the generic braid compiler.

In [ ]:
named_trefoil = KnotFunction.from_name("trefoil")
named_figure8 = KnotFunction.from_name("figure_eight")
figure8_braid = KnotFunction.from_name(
    "4_1", construction="braid",
    validation_samples=512 if QUICK else 2048,
)

for f in (named_trefoil, named_figure8, figure8_braid):
    print(
        f.name,
        "components =", f.expected_components,
        "construction =", f.metadata.get("construction"),
        "catalogue =", f.metadata.get("catalogue_name"),
    )


## 9. Custom analytic fields from Python or SymPy

The common `KnotFunction` interface is not restricted to the built-in knot constructors.

In [ ]:
def custom_callable(x, y, z):
    return (x*x + y*y + z*z - 1.0) + 1j*z

custom_field = KnotFunction.from_function(
    custom_callable, name="custom_callable_example"
)

x, y, z = sp.symbols("x y z", real=True)
custom_expr = (x**2 + y**2 + z**2 - 1) + sp.I*z
sympy_field = KnotFunction.from_function(
    custom_expr, symbols=(x,y,z), name="custom_sympy_example"
)

print(custom_field(np.array([0.,1.,0.5]), 0., 0.))
display(sympy_field.symbolic_r3_expression((x,y,z)))


## 10. Sample the field in 3D

For extraction we sample \(F\) on a Cartesian grid. We use the direct trefoil field
\(u^2-v^3\) for a fast, clean demonstration.

In [ ]:
analysis_field = trefoil
SPAN = ((-4.0,4.0),)*3

sample = analysis_field.sample(span=SPAN, dimension=GRID_DIM)
print("shape:", sample.values.shape)
print("origin:", sample.origin)
print("spacing:", sample.spacing)
print("|F| range:", (sample.abs_values.min(), sample.abs_values.max()))

mid = sample.values.shape[2]//2
plt.figure(figsize=(6,5))
plt.imshow(
    sample.abs_values[:,:,mid].T,
    origin="lower",
    extent=(SPAN[0][0],SPAN[0][1],SPAN[1][0],SPAN[1][1]),
    aspect="equal",
)
plt.xlabel("x"); plt.ylabel("y")
plt.title(r"$|F(x,y,z=0)|$")
plt.colorbar(label=r"$|F|$")
plt.show()


## 11. Tubular neighborhoods and topology diagnostics

We study


$$
\Omega_\epsilon=\{|F|\le\epsilon\},
$$


whose boundary is \(|F|=\epsilon\). Rather than assuming a radius is valid, scan
candidate radii and inspect:
- volume component count,
- surface component count,
- Euler characteristic,
- closedness,
- contact with the finite box boundary,
- total boundary genus,
- agreement with the expected tubular-neighborhood topology.

In [ ]:
candidate_radii = np.array([0.16, 0.20, 0.25, 0.30, 0.35, 0.40])
diagnostics = []

for eps in candidate_radii:
    try:
        d = analysis_field.diagnose_level(
            float(eps), sample=sample, span=SPAN, dimension=GRID_DIM
        )
        diagnostics.append(d)
        print(
            f"eps={eps:.3f} | voxels={d.occupied_voxels:6d} | "
            f"Vcomp={d.volume_components} | Scomp={d.surface_components} | "
            f"chi={d.surface_euler_characteristic} | closed={d.surface_is_closed} | "
            f"boundary={d.touches_box_boundary} | genus={d.total_boundary_genus} | "
            f"expected_match={d.matches_expected_tubular_neighborhood}"
        )
    except Exception as exc:
        print(f"eps={eps:.3f}: {type(exc).__name__}: {exc}")

valid = [d for d in diagnostics if d.matches_expected_tubular_neighborhood is True]
EPS = valid[0].radius if valid else None
print("\nSelected EPS:", EPS)


If no radius passes at quick resolution, increase `GRID_DIM`, enlarge `SPAN`, or refine
the radius grid. The notebook deliberately does not silently treat a failed diagnostic
as a valid topology.

## 12. Sublevel mask and level surface

In [ ]:
if EPS is not None:
    mask, _ = analysis_field.sublevel_mask(
        EPS, sample=sample, span=SPAN, dimension=GRID_DIM, require_compact=True
    )
    print("occupied voxels:", np.count_nonzero(mask))

    mesh = analysis_field.level_surface(
        EPS, sample=sample, span=SPAN, dimension=GRID_DIM, require_compact=True
    )
    print("mesh vertices:", mesh.vertices.shape)
    print("mesh faces:", mesh.faces.shape)

    fig = plt.figure(figsize=(6.2, 5.6))
    ax = fig.add_subplot(111, projection="3d")
    plot_level_surface_mesh(ax, mesh, alpha=0.42)
    set_axes_equal_3d(ax, mesh.vertices)
    style_3d_axis(ax, labels=True)
    ax.set_title(rf"Trefoil level surface $|F|={EPS:.2f}$")
    save_paper_figure(fig, "05_trefoil_level_surface")
    plt.show()
else:
    print("No certified EPS at this resolution. Try GRID_DIM >= 56 and include epsilon=0.30.")


## 13. Resolution convergence

A single grid should not be treated as a numerical certificate. The convergence helper
repeats the topology diagnostics over several resolutions and compares their signatures.

In [ ]:
if EPS is not None:
    conv = analysis_field.tubular_convergence(
        EPS, dimensions=CONVERGENCE_DIMS, span=SPAN
    )
    print("converged:", conv.converged)
    for d in conv.diagnostics:
        print(d)


## 14. Tubular volume \(\rightarrow\) spatial graph

The analytic field is now connected directly to the existing extraction pipeline:


$$
|F|\le\epsilon
\rightarrow \text{voxel volume}
\rightarrow \text{skeleton}
\rightarrow \text{embedded MultiGraph}.
$$



In [ ]:
graph = None
if EPS is not None:
    graph = analysis_field.to_spatial_graph(
        EPS, sample=sample, span=SPAN, dimension=GRID_DIM
    )
    print("nodes:", graph.number_of_nodes())
    print("edges:", graph.number_of_edges())
    print("connected components:", nx.number_connected_components(graph))
    print("degree sequence:", sorted((d for _,d in graph.degree()), reverse=True))
    print("metadata:", graph.graph)


In [ ]:
if graph is not None and graph.number_of_edges():
    fig = plt.figure(figsize=(6.2, 5.6))
    ax = fig.add_subplot(111, projection="3d")
    graph_pts = plot_spatial_graph_3d(ax, graph)
    set_axes_equal_3d(ax, graph_pts)
    style_3d_axis(ax, labels=True)
    ax.set_title("Extracted trefoil spatial graph")
    save_paper_figure(fig, "05_trefoil_spatial_graph")
    plt.show()


## 15. Yamada polynomial

Once extraction returns the ordinary embedded `MultiGraph`, the existing Yamada pipeline
can be used without a separate knot-field-specific invariant implementation.

This uses the shared `YAMADA_OPTIONS` projection settings. If a single extracted skeleton is projection-degenerate, the error is reported and the dense phase scan still continues with error phases.


In [ ]:
if RUN_YAMADA and graph is not None:
    from knotted_graph.invariants.yamada import compute_graph_yamada_polynomial
    from knotted_graph.projection import compute_yamada_polynomial
    A = sp.Symbol("A")
    try:
        yamada = compute_yamada_polynomial(graph, A, **YAMADA_OPTIONS)
        display(sp.expand(yamada))
    except Exception as exc:
        yamada = None
        print(f"Yamada projection failed for this extracted graph: {type(exc).__name__}: {exc}")
else:
    print("Set RUN_YAMADA = True to run Yamada evaluation.")


## 16. Gauge-fixed paths between analytic knot fields

For endpoints \(F_0,F_1\), `KnotFunctionPath` supports RMS normalization and global
phase alignment before interpolation:


$$
F_\lambda=(1-\lambda)\tilde F_0+\lambda e^{i\phi}\tilde F_1.
$$


This removes trivial scale/phase mismatch, but the path is not claimed to be canonical.

In [ ]:
start_field = KnotFunction.from_name("trefoil")
end_field = KnotFunction.from_name("figure_eight")

path = KnotFunctionPath(
    start_field, end_field,
    normalize=True, phase_align=True,
    sample_count=2048 if PAPER_FIGURE_MODE else (1024 if QUICK else 4096),
    seed=23,
)
print(path.gauge)

for lam in (0.,0.25,0.5,0.75,1.):
    f = path.at(lam)
    print(lam, f.name, f.metadata.get("construction"))


## 17A. Wide level-set topology scan in $(\lambda,\epsilon)$
This scan is intentionally separate from Yamada.  Yamada is a graph/core invariant, while large level radii change the topology of the sublevel handlebody itself.  The wide scan uses a larger box and classifies the level surface by connected components, Euler characteristic, closedness, boundary contact, and total genus.

The vertical coordinate is the analytic threshold $|F_\lambda|\le\epsilon$ after gauge normalization, not a Euclidean tube radius.  Large $\epsilon$ therefore need not move monotonically from knot tube to sphere: critical values of $|F_\lambda|$ can create higher-genus surfaces, and in the stereographic $\mathbb{R}^3$ chart the sublevel set becomes noncompact once $\epsilon$ reaches the projection-pole value.  The dashed line marks this projection-pole threshold.  Orange `multi-component` cells are not promoted to sphere/tube claims; they indicate that the sampled volume or boundary has more than one component, which can be real topology or finite-grid fragmentation in small/complex levels.  Green `sphere` cells are assigned only when the sampled volume and boundary are both single-component and the closed boundary has total genus 0.


In [ ]:
topology_scan_result = None
if RUN_TOPOLOGY_PHASE_SCAN:
    topology_scan_result = cached_topology_phase_scan(
        "trefoil_to_figure8_levelset_topology_wide",
        path,
        lambdas=TOPOLOGY_PHASE_LAMBDAS,
        radii=TOPOLOGY_PHASE_RADII,
        dimension=TOPOLOGY_PHASE_DIM,
        span=TOPOLOGY_SPAN,
    )
    pole_abs = path_projection_pole_abs(path, TOPOLOGY_PHASE_LAMBDAS)
    print("topology records:", len(topology_scan_result.records))
    print("topology phase counts:", Counter(record.phase_signature for record in topology_scan_result.records))
    print("projection pole abs range:", (float(np.nanmin(pole_abs)), float(np.nanmax(pole_abs))))
    print("endpoint topology phases:")
    for r in topology_scan_result.records:
        if np.isclose(r.lam, 0.0) or np.isclose(r.lam, 1.0):
            phase = compact_phase_label(r.phase_signature)
            print(
                f"lambda={r.lam:.3f} eps={r.radius:.3f} phase={phase} "
                f"V={r.volume_components} S={r.surface_components} "
                f"chi={r.surface_euler_characteristic} genus={r.total_boundary_genus} "
                f"closed={r.surface_is_closed} boundary={r.touches_box_boundary} error={r.error}"
            )
    topology_transitions = topology_scan_result.transition_points()
    print_transition_summary(topology_transitions, heading="Topology transition intervals", limit=40)
else:
    print("Set RUN_TOPOLOGY_PHASE_SCAN=True to run the wide level-set topology scan.")


In [ ]:
if topology_scan_result is not None:
    ax, topology_legend_rows = plot_phase_signature_grid(
        topology_scan_result,
        title="Trefoil to figure-eight level-set topology phases",
    )
    plot_projection_pole_threshold(ax, path, TOPOLOGY_PHASE_LAMBDAS)
    save_paper_figure(ax.figure, "05_trefoil_figure8_levelset_topology_phase_signatures")
    plt.show()
    print_legend_rows(topology_legend_rows)

    fig, axes = plt.subplots(1, 3, figsize=(13.0, 3.8), constrained_layout=True)
    plot_topology_observable(topology_scan_result, "volume_components", title="volume components", ax=axes[0])
    plot_topology_observable(topology_scan_result, "surface_components", title="surface components", ax=axes[1])
    plot_topology_observable(topology_scan_result, "total_boundary_genus", title="total genus", ax=axes[2])
    save_paper_figure(fig, "05_trefoil_figure8_levelset_topology_observables")
    plt.show()


### Representative topology convergence audit
The phase map is a finite-grid atlas.  The audit below reruns representative cells across several grid dimensions.  It checks that stable claims such as `knot/link tube`, `sphere`, `box boundary`, and `noncompact pole` do not depend on a single resolution.  Exact genus values inside `higher genus` are not promoted to phase labels unless they converge; the phase map groups them because the broad topology class is more stable than the integer genus near critical levels.


In [ ]:
topology_convergence_audit_df = None
if topology_scan_result is not None and RUN_TOPOLOGY_CONVERGENCE_AUDIT:
    import pandas as pd

    audit_cases = [
        ("tube_endpoint", 0.0, 0.7941176470588235),
        ("sphere_window", 0.75, 0.7594117647058823),
        ("higher_genus_mid", 0.50, 0.5164705882352941),
        ("near_pole_mid", 0.29166666666666663, 1.0370588235294118),
        ("boundary_endpoint", 0.0, 1.2452941176470589),
    ]
    audit_dims = (112, 144, 176)

    def classify_topology_diagnostic(diagnostic):
        if diagnostic.touches_box_boundary:
            return "box boundary"
        if not diagnostic.surface_is_closed or diagnostic.total_boundary_genus is None:
            return "open surface"
        if diagnostic.volume_components != 1 or diagnostic.surface_components != 1:
            return "multi-component"
        if diagnostic.total_boundary_genus == 0:
            return "sphere"
        if diagnostic.total_boundary_genus == 1:
            return "knot/link tube"
        return "higher genus"

    rows = []
    for case, lam, eps in audit_cases:
        field = path.at(float(lam))
        pole = field.projection_pole_value
        pole_abs = None if pole is None else float(abs(pole))
        if pole_abs is not None and eps >= pole_abs:
            rows.append({
                "case": case, "lambda": lam, "epsilon": eps,
                "dimension": "analytic", "phase": "noncompact pole",
                "volume_components": None, "surface_components": None,
                "chi": None, "genus": None, "closed": None,
                "boundary": None, "pole_abs": pole_abs,
            })
            continue
        for dimension in audit_dims:
            sample = field.sample(span=TOPOLOGY_SPAN, dimension=dimension)
            diagnostic = field.diagnose_level(
                eps, sample=sample, span=TOPOLOGY_SPAN, dimension=dimension
            )
            rows.append({
                "case": case, "lambda": lam, "epsilon": eps,
                "dimension": dimension,
                "phase": classify_topology_diagnostic(diagnostic),
                "volume_components": diagnostic.volume_components,
                "surface_components": diagnostic.surface_components,
                "chi": diagnostic.surface_euler_characteristic,
                "genus": diagnostic.total_boundary_genus,
                "closed": diagnostic.surface_is_closed,
                "boundary": diagnostic.touches_box_boundary,
                "pole_abs": pole_abs,
            })
    topology_convergence_audit_df = pd.DataFrame(rows)
    display(topology_convergence_audit_df)
else:
    print("Set RUN_TOPOLOGY_CONVERGENCE_AUDIT=True to run representative topology convergence checks.")


## 17B. Full spatial-graph Yamada scan in \((\lambda,\epsilon)\)
This scan evaluates Yamada on the reduced compact graph, not only on closed knot/link cores.  Before Yamada evaluation the graph removes leaf artifacts and suppresses degree-2 subdivision vertices with `simplify_edges`, so unnecessary sampled vertices along the same arc do not change the invariant.  The wide topology scan is used as a sphere/ball oracle: nearby single closed genus-0 cells are assigned the isolated-vertex Yamada value, avoiding spurious loop skeletons inside sphere/ball regions without repeating expensive topology diagnostics.  Closed knot/link cores and small branched cores use embedded spatial-graph Yamada.  Larger branched cyclic cores use the library's fast crossing-free graph-Yamada fallback and are marked with `G` phase labels, because exhaustive projection/crossing search can dominate the notebook runtime.  Boundary-contacting regions and projection-degenerate diagrams are still shown as non-certified phases.


In [ ]:
scan_result = None
if RUN_DEFORMATION_SCAN:
    phase_kind = "yamada" if RUN_YAMADA else "graph"
    scan_result = cached_deformation_scan(
        f"trefoil_to_figure8_{phase_kind}_certified",
        path,
        lambdas=GALLERY_LAMBDAS,
        radii=GALLERY_PHASE_RADII,
        dimension=GALLERY_PHASE_DIM,
        invariant="yamada" if RUN_YAMADA else None,
        yamada_options=YAMADA_OPTIONS if RUN_YAMADA else None,
    )
    print("records:", len(scan_result.records))
    phase_family_counts = Counter(record.phase_signature.split(":", 1)[0] for record in scan_result.records)
    error_category_counts = Counter(
        normalize_scan_error(record.error) if record.error else "ok"
        for record in scan_result.records
    )
    print("valid Yamada records:", sum(record.yamada is not None for record in scan_result.records))
    print("phase family counts:", phase_family_counts)
    print("error category counts:", error_category_counts)
    if error_category_counts.get("core", 0):
        raise AssertionError("core-rejection errors remain in the Yamada phase scan")
    record_labels = phase_label_lookup(scan_result)
    endpoint_records = [
        record for record in scan_result.records
        if np.isclose(record.lam, 0.0) or np.isclose(record.lam, 1.0)
    ]
    print("endpoint phase counts:")
    for endpoint in sorted({round(float(record.lam), 10) for record in endpoint_records}):
        endpoint_counts = Counter(
            compact_phase_label(record.phase_signature, record_labels)
            for record in endpoint_records
            if np.isclose(record.lam, endpoint)
        )
        print(f"lambda={endpoint:.3f}", dict(endpoint_counts))
    print("first records:")
    for r in scan_result.records[:10]:
        print({
            "lambda": round(r.lam, 3),
            "epsilon": round(r.radius, 3),
            "nodes": r.nodes,
            "edges": r.edges,
            "phase": compact_phase_label(r.phase_signature, record_labels),
            "error": normalize_scan_error(r.error) if r.error else None,
        })

    transition_intervals = scan_result.transition_points()
    print_transition_summary(
        transition_intervals,
        yamada_labels=phase_label_lookup(scan_result),
        heading="Transition intervals",
        limit=40,
    )


In [ ]:
if scan_result is not None:
    ax, phase_legend_rows = plot_phase_signature_grid(
        scan_result,
        title=f"Trefoil to figure-eight {phase_kind} phase signatures",
    )
    save_paper_figure(ax.figure, f"05_trefoil_figure8_{phase_kind}_phase_signatures")
    plt.show()
    print_legend_rows(phase_legend_rows)

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), constrained_layout=True)
    plot_phase_observable(scan_result, "nodes", title="nodes", ax=axes[0])
    plot_phase_observable(scan_result, "cycle_rank", title="cycle rank", ax=axes[1])
    save_paper_figure(fig, f"05_trefoil_figure8_{phase_kind}_scan_graph_observables")
    plt.show()


### Compactified high-radius Yamada limit

The finite R3 chart becomes unreliable once a level radius includes the stereographic projection pole.  For semiholomorphic fields on S3, however, `sum(abs(coefficients))` is a certified upper bound for `|F|` because `|u|, |v|, |vbar| <= 1`.  Any radius above that bound fills the compact S3 model, so the phase is assigned the isolated-vertex graph with Yamada polynomial `-1`.


In [ ]:
compact_scan_result = None
compact_overview_result = None
if RUN_DEFORMATION_SCAN and RUN_YAMADA:
    s3_bounds = np.asarray([
        semiholomorphic_s3_abs_upper_bound(path.at(float(lam)))
        for lam in GALLERY_LAMBDAS
    ], dtype=float)
    if not np.all(np.isfinite(s3_bounds)):
        raise AssertionError("compactified scan requires finite semiholomorphic S3 bounds")
    compact_radius_max = float(np.max(s3_bounds) * (1.0 + S3_VERTEX_BOUND_MARGIN) * 1.05)
    compact_high_radii = np.geomspace(max(float(GALLERY_PHASE_RADII[-1]) * 1.15, 0.42), compact_radius_max, 26)
    compact_phase_radii = np.unique(np.round(np.concatenate([GALLERY_PHASE_RADII, compact_high_radii]), 10))
    print("S3 |F| certified upper-bound range:", (float(np.min(s3_bounds)), float(np.max(s3_bounds))))
    print("compactified radii:", (float(compact_phase_radii[0]), float(compact_phase_radii[-1]), len(compact_phase_radii)))
    compact_scan_result = cached_deformation_scan(
        "trefoil_to_figure8_yamada_compactified_certified",
        path,
        lambdas=GALLERY_LAMBDAS,
        radii=compact_phase_radii,
        dimension=GALLERY_PHASE_DIM,
        invariant="yamada",
        yamada_options=YAMADA_OPTIONS,
    )
    compact_labels = phase_label_lookup(compact_scan_result)
    compact_error_counts = Counter(
        normalize_scan_error(record.error) if record.error else "ok"
        for record in compact_scan_result.records
    )
    compact_phase_counts = Counter(
        compact_phase_label(record.phase_signature, compact_labels)
        for record in compact_scan_result.records
    )
    top_radius = float(compact_phase_radii[-1])
    top_records = [record for record in compact_scan_result.records if np.isclose(record.radius, top_radius)]
    top_phase_counts = Counter(record.phase_signature for record in top_records)
    print("compactified records:", len(compact_scan_result.records))
    print("compactified error counts:", compact_error_counts)
    print("compactified phase counts:", compact_phase_counts)
    print("top-radius phase counts:", top_phase_counts)
    if set(top_phase_counts) != {"yamada:contractible-vertex"}:
        raise AssertionError("highest radius is not certified as the all-S3 / vertex Yamada phase")

    compact_overview_result = phase_overview_result(compact_scan_result)
    ax, compact_overview_rows = plot_phase_signature_grid(
        compact_overview_result,
        title="Trefoil to figure-eight compactified Yamada certification overview",
    )
    ax.set_yscale("log")
    save_paper_figure(ax.figure, "05_trefoil_to_figure8_yamada_compactified_overview")
    plt.show()
    print_legend_rows(compact_overview_rows, limit=12)

    ax, compact_exact_rows = plot_phase_signature_grid(
        compact_scan_result,
        title="Trefoil to figure-eight compactified exact Yamada signatures",
    )
    ax.set_yscale("log")
    save_paper_figure(ax.figure, "05_trefoil_to_figure8_yamada_compactified_phase_signatures")
    plt.show()
    print_legend_rows(compact_exact_rows, limit=20)


### Five-transition compactified Yamada gallery

The compactified certificate above is not tied to the trefoil-to-figure-eight path.  The gallery below applies the same scan, core simplification, Yamada evaluation, chart-gap classification, and S3 vertex bound to five independent knot/link transitions.  Boundary and chart-gap cells are displayed as certified non-Yamada regimes; the cell asserts that no core, projection, topology, or numerical computation errors remain and that every top-radius lambda is the isolated-vertex Yamada phase.


In [ ]:
if "show_rows" not in globals():
    def show_rows(rows, limit=10):
        try:
            import pandas as pd
            display(pd.DataFrame(rows).head(limit))
        except Exception:
            for row in rows[:limit]:
                print(row)


COMPACTIFIED_GALLERY_SPECS = [
    (
        "hopf_to_trefoil",
        lambda: KnotFunction.from_name("hopf_link"),
        lambda: KnotFunction.from_name("trefoil"),
        "Hopf link to trefoil",
    ),
    (
        "hopf_to_solomon",
        lambda: KnotFunction.from_name("hopf_link"),
        lambda: KnotFunction.from_name("solomon_link"),
        "Hopf link to Solomon link",
    ),
    (
        "trefoil_to_cinquefoil",
        lambda: KnotFunction.from_name("trefoil"),
        lambda: KnotFunction.from_name("cinquefoil"),
        "Trefoil to cinquefoil",
    ),
    (
        "figure8_to_hopf",
        lambda: KnotFunction.from_name("figure_eight"),
        lambda: KnotFunction.from_name("hopf_link"),
        "Figure-eight to Hopf link",
    ),
    (
        "unknot_to_solomon",
        lambda: KnotFunction.from_name("unknot"),
        lambda: KnotFunction.from_name("solomon_link"),
        "Unknot to Solomon link",
    ),
]

compactified_gallery_results = []
compactified_gallery_rows = []
if RUN_DEFORMATION_SCAN and RUN_YAMADA:
    compactified_gallery_yamada_options = dict(YAMADA_OPTIONS)
    compactified_gallery_yamada_options["num_rotation_samples"] = max(
        int(compactified_gallery_yamada_options.get("num_rotation_samples", 0)),
        64,
    )
    for label, start_factory, end_factory, title in COMPACTIFIED_GALLERY_SPECS:
        start = start_factory()
        end = end_factory()
        gallery_path = KnotFunctionPath(
            start, end,
            normalize=True,
            phase_align=True,
            sample_count=2048 if PAPER_FIGURE_MODE else (1024 if QUICK else 4096),
            seed=23,
        )
        print(f"{label}: {start.name} -> {end.name}")
        print("gauge:", gallery_path.gauge)
        exact_result, overview_result, radii, s3_bounds, audit = run_compactified_yamada_transition(
            label,
            gallery_path,
            lambdas=GALLERY_LAMBDAS,
            base_radii=GALLERY_PHASE_RADII,
            dimension=GALLERY_PHASE_DIM,
            yamada_options=compactified_gallery_yamada_options,
        )
        compactified_gallery_results.append({
            "label": label,
            "title": title,
            "path": gallery_path,
            "exact": exact_result,
            "overview": overview_result,
            "radii": radii,
            "s3_bounds": s3_bounds,
            "audit": audit,
        })
        compactified_gallery_rows.append({
            "transition": label,
            "start": start.name,
            "end": end.name,
            "lambda_samples": len(GALLERY_LAMBDAS),
            "radius_samples": len(radii),
            "records": audit["records"],
            "valid_yamada": audit["valid_yamada"],
            "vertex_records": audit["vertex_records"],
            "top_radius": audit["top_radius"],
            "top_vertex_lambdas": audit["top_phase_counts"].get("yamada:contractible-vertex", 0),
            "bad_errors": dict(audit["bad_error_counts"]),
            "allowed_non_yamada": {
                key: audit["error_counts"][key]
                for key in ("boundary", "chart-gap")
                if audit["error_counts"].get(key, 0)
            },
        })

        ax, exact_rows = plot_phase_signature_grid(
            exact_result,
            title=f"{title}: compactified exact Yamada signatures",
        )
        ax.set_yscale("log")
        save_paper_figure(ax.figure, f"05_{label}_compactified_yamada_exact_phase_signatures")
        plt.show()
        print_legend_rows(exact_rows, limit=16)

    gallery_fig, gallery_axes = plot_compactified_overview_gallery(
        compactified_gallery_results,
        title="Five compactified Yamada transition certificates",
    )
    save_paper_figure(gallery_fig, "05_compactified_yamada_five_transition_overview")
    plt.show()

show_rows(compactified_gallery_rows, limit=10)


## 18. Use-case gallery: generating findings and phase diagrams
The previous sections walk one carefully validated example through the full pipeline.  The cells below turn the same API into reusable finding generators: exact braid audits, sampled compiler validation tables, epsilon-acceptance maps, optional graph-observable summaries, and deformation phase diagrams.

For speed, `FAST_INTERACTIVE=True` runs only the exact and compiler galleries by default.  Turn on `RUN_GALLERY_DIAGNOSTICS`, `RUN_GALLERY_PHASE_ATLAS`, `RUN_GALLERY_GRAPH_EXTRACTION`, or `RUN_GALLERY_CONVERGENCE` selectively when generating final figures or scientific validation tables.

Keep the claim boundary the same as above.  Exact rows are combinatorial statements about the supplied representation.  Heat maps and phase diagrams are numerical observations at the displayed span, grid dimension, radius grid, and chosen function-space path.


In [ ]:
def show_rows(rows, *, limit=None):
    rows = list(rows)
    if limit is not None:
        rows = rows[:limit]
    if not rows:
        print("No rows to display.")
        return
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)


def diagnostic_row(label, diagnostic, *, claim=""):
    row = dataclasses.asdict(diagnostic)
    row.update({
        "use_case": label,
        "claim_template": claim,
        "epsilon": float(diagnostic.radius),
        "status": "accepted" if diagnostic.matches_expected_tubular_neighborhood is True else "not_accepted",
        "error": None,
    })
    return row


def diagnostic_error_row(label, eps, exc, *, claim=""):
    return {
        "use_case": label,
        "claim_template": claim,
        "epsilon": float(eps),
        "radius": float(eps),
        "occupied_voxels": None,
        "volume_components": None,
        "surface_components": None,
        "surface_euler_characteristic": None,
        "surface_is_closed": None,
        "touches_box_boundary": None,
        "total_boundary_genus": None,
        "expected_components": None,
        "matches_expected_tubular_neighborhood": None,
        "status": "error",
        "error": f"{type(exc).__name__}: {exc}",
    }


def scan_level_radii(field, label, radii, *, span, dimension, claim=""):
    sample = field.sample(span=span, dimension=dimension)
    rows = []
    diagnostics = []
    for eps in radii:
        try:
            diagnostic = field.diagnose_level(
                float(eps), sample=sample, span=span, dimension=dimension
            )
            diagnostics.append(diagnostic)
            rows.append(diagnostic_row(label, diagnostic, claim=claim))
        except Exception as exc:
            rows.append(diagnostic_error_row(label, eps, exc, claim=claim))
    return {"field": field, "sample": sample, "diagnostics": diagnostics, "rows": rows}


def _acceptance_code(row):
    if row.get("error"):
        return -1
    if row.get("matches_expected_tubular_neighborhood") is True:
        return 1
    return 0


def plot_acceptance_grid(rows, *, title="Epsilon acceptance map"):
    rows = list(rows)
    labels = list(dict.fromkeys(row["use_case"] for row in rows))
    radii = sorted({float(row["epsilon"]) for row in rows})
    if not labels or not radii:
        print("No acceptance grid to plot.")
        return None
    matrix = np.full((len(labels), len(radii)), -1, dtype=int)
    label_index = {label: i for i, label in enumerate(labels)}
    radius_index = {radius: j for j, radius in enumerate(radii)}
    for row in rows:
        matrix[label_index[row["use_case"]], radius_index[float(row["epsilon"])] ] = _acceptance_code(row)

    cmap = mcolors.ListedColormap(["#c44e52", "#d8d8d8", "#55a868"])
    norm = mcolors.BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap.N)
    fig, ax = plt.subplots(figsize=(max(7, 0.85 * len(radii)), max(3, 0.55 * len(labels))))
    image = ax.imshow(matrix, aspect="auto", interpolation="nearest", cmap=cmap, norm=norm)
    ax.set_xticks(np.arange(len(radii)))
    ax.set_xticklabels([f"{radius:.2f}" for radius in radii])
    ax.set_yticks(np.arange(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xlabel(r"level radius $\epsilon$")
    ax.set_title(title)
    cbar = fig.colorbar(image, ax=ax, ticks=[-1, 0, 1], shrink=0.82)
    cbar.ax.set_yticklabels(["error", "not accepted", "accepted"])
    fig.tight_layout()
    return fig, ax


def phase_summary_rows(result, label):
    rows = []
    for record in result.records:
        rows.append({
            "use_case": label,
            "lambda": record.lam,
            "epsilon": record.radius,
            "nodes": record.nodes,
            "edges": record.edges,
            "components": record.components,
            "cycle_rank": record.cycle_rank,
            "degree_sequence": record.degree_sequence,
            "phase_signature": record.phase_signature,
            "yamada": None if record.yamada is None else sp.sstr(sp.expand(record.yamada)),
            "error": record.error,
        })
    return rows


def plot_phase_observable(result, observable, *, title=None, ax=None):
    grid = result.record_grid()
    values = np.empty(grid.shape, dtype=float)
    for row_index, records in enumerate(grid):
        for column_index, record in enumerate(records):
            values[row_index, column_index] = np.nan if record.error else float(getattr(record, observable))
    if ax is None:
        _, ax = plt.subplots()
    image = ax.imshow(
        values,
        origin="lower",
        aspect="auto",
        extent=(
            float(result.lambdas[0]), float(result.lambdas[-1]),
            float(result.radii[0]), float(result.radii[-1]),
        ),
        interpolation="nearest",
    )
    ax.set_xlabel(r"$\lambda$")
    ax.set_ylabel(r"level radius $\epsilon$")
    ax.set_title(title or observable.replace("_", " "))
    plt.colorbar(image, ax=ax, shrink=0.82)
    return ax


### 18.1 Exact catalogue and braid-closure audit
This table is a cheap regression guard and a compact finding: the catalogue entries have the advertised closure component counts for their supplied braid representatives.  The strand count is still representation-local; it is not the global braid index.


In [ ]:
catalogue_audit_rows = []
if RUN_USE_CASE_GALLERY:
    for name in available_knot_names():
        entry = get_knot_entry(name)
        required_strands = infer_braid_strands(entry.braid_word, entry.strands)
        permutation = braid_permutation(entry.braid_word, entry.strands)
        components = braid_component_count(entry.braid_word, entry.strands)
        assert required_strands == entry.strands
        assert components == entry.components
        catalogue_audit_rows.append({
            "name": entry.canonical_name,
            "aliases": entry.aliases,
            "braid_word": entry.braid_word,
            "strands_for_this_word": required_strands,
            "permutation": permutation,
            "closure_components": components,
            "torus_params": entry.torus_params,
            "reference_field": entry.reference_field,
        })

show_rows(catalogue_audit_rows)


### 18.2 Arbitrary-braid compiler gallery
These examples show the generic Artin-braid route beyond a single figure-eight.  The `passed` column is sampled numerical validation of the finite Fourier approximation, not a theorem-level all-scale certificate.


In [ ]:
BRAID_GALLERY = [
    {"label": "trefoil braid", "word": (1, 1, 1), "expected_components": 1},
    {"label": "figure-eight braid", "word": (1, -2, 1, -2), "expected_components": 1},
    {"label": "Hopf link braid", "word": (1, 1), "expected_components": 2},
    {"label": "Borromean rings braid", "word": (1, -2, 1, -2, 1, -2), "expected_components": 3},
]

braid_gallery_rows = []
if RUN_USE_CASE_GALLERY and RUN_GALLERY_COMPILER:
    for case in BRAID_GALLERY:
        label, word = case["label"], case["word"]
        strands_for_word = infer_braid_strands(word)
        components = braid_component_count(word, strands_for_word)
        assert components == case["expected_components"]
        row = {
            "use_case": label,
            "word": word,
            "strands_for_this_word": strands_for_word,
            "closure_components": components,
            "expected_components": case["expected_components"],
        }
        try:
            polynomial, report = braid_to_semiholomorphic(
                word,
                strands=strands_for_word,
                validation_samples=256 if QUICK else 1024,
                fourier_modes=(4, 8, 12, 16, 24, 32) if QUICK else (4, 8, 12, 16, 24, 32, 48, 64),
            )
            assert report.passed
            row.update({
                "passed": report.passed,
                "fourier_mode": report.fourier_mode,
                "term_count": len(polynomial.terms),
                "max_root_error": report.max_root_error,
                "min_target_separation": report.min_target_separation,
                "error_fraction": report.error_fraction,
                "interpretation": report.interpretation,
                "error": None,
            })
        except Exception as exc:
            row.update({
                "passed": False,
                "fourier_mode": None,
                "term_count": None,
                "max_root_error": None,
                "min_target_separation": None,
                "error_fraction": None,
                "interpretation": "failed sampled compiler validation",
                "error": f"{type(exc).__name__}: {exc}",
            })
        braid_gallery_rows.append(row)

show_rows(braid_gallery_rows)


### 18.3 Static epsilon-acceptance atlas
This atlas scans several knot/link fields over the same candidate radii.  Accepted cells mean the sampled sublevel set passed the library diagnostics at this grid: nonempty, closed, off the box boundary, and matching the expected component/genus signature.  Rejected cells are useful too; they identify radii or resolutions that should not be used for a topology claim.


In [ ]:
STATIC_USE_CASES = [
    ("unknot T(1,1)", lambda: KnotFunction.torus(1, 1), "baseline single-component tube"),
    ("trefoil T(2,3)", lambda: KnotFunction.torus(2, 3), "principal nontrivial knot tube"),
    ("Hopf link T(2,2)", lambda: KnotFunction.torus(2, 2), "two-component link tube"),
    ("cinquefoil T(2,5)", lambda: KnotFunction.torus(2, 5), "higher winding torus knot tube"),
]

gallery_diagnostic_results = {}
gallery_diagnostic_rows = []
if RUN_USE_CASE_GALLERY and RUN_GALLERY_DIAGNOSTICS:
    for label, factory, claim in STATIC_USE_CASES:
        result = scan_level_radii(
            factory(), label, GALLERY_RADII,
            span=SPAN, dimension=GALLERY_DIM, claim=claim,
        )
        gallery_diagnostic_results[label] = result
        gallery_diagnostic_rows.extend(result["rows"])

show_rows(gallery_diagnostic_rows)
if gallery_diagnostic_rows:
    fig_ax = plot_acceptance_grid(
        gallery_diagnostic_rows,
        title=f"Static knot/link epsilon acceptance at dimension {GALLERY_DIM}",
    )
    if fig_ax is not None:
        save_paper_figure(fig_ax[0], "05_static_epsilon_acceptance_map")
    plt.show()


### 18.4 Optional gallery graph observables
Enable `RUN_GALLERY_GRAPH_EXTRACTION` to turn every accepted static tube into an embedded `networkx.MultiGraph` and compare graph observables.  This is kept off by default because skeletonization can dominate runtime when many use cases are scanned.


In [ ]:
gallery_graph_rows = []
if RUN_USE_CASE_GALLERY and RUN_GALLERY_GRAPH_EXTRACTION:
    for label, result in gallery_diagnostic_results.items():
        accepted = [
            row for row in result["rows"]
            if row.get("matches_expected_tubular_neighborhood") is True
        ]
        if not accepted:
            gallery_graph_rows.append({
                "use_case": label,
                "epsilon": None,
                "nodes": None,
                "edges": None,
                "components": None,
                "cycle_rank": None,
                "degree_sequence": None,
                "error": "no accepted epsilon in static atlas",
            })
            continue
        eps = float(accepted[0]["epsilon"])
        try:
            graph = result["field"].to_spatial_graph(
                eps, sample=result["sample"], span=SPAN, dimension=GALLERY_DIM
            )
            components = nx.number_connected_components(graph) if graph.number_of_nodes() else 0
            cycle_rank = graph.number_of_edges() - graph.number_of_nodes() + components
            gallery_graph_rows.append({
                "use_case": label,
                "epsilon": eps,
                "nodes": graph.number_of_nodes(),
                "edges": graph.number_of_edges(),
                "components": components,
                "cycle_rank": cycle_rank,
                "degree_sequence": tuple(sorted((degree for _, degree in graph.degree()), reverse=True)),
                "error": None,
            })
        except Exception as exc:
            gallery_graph_rows.append({
                "use_case": label,
                "epsilon": eps,
                "nodes": None,
                "edges": None,
                "components": None,
                "cycle_rank": None,
                "degree_sequence": None,
                "error": f"{type(exc).__name__}: {exc}",
            })

show_rows(gallery_graph_rows)
valid_graph_rows = [row for row in gallery_graph_rows if row.get("error") is None]
if valid_graph_rows:
    labels = [row["use_case"] for row in valid_graph_rows]
    xloc = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(8.2, 4.2), constrained_layout=True)
    width = 0.22
    ax.bar(xloc - width, [row["nodes"] for row in valid_graph_rows], width, label="nodes", color="#4c78a8")
    ax.bar(xloc, [row["edges"] for row in valid_graph_rows], width, label="edges", color="#f58518")
    ax.bar(xloc + width, [row["cycle_rank"] for row in valid_graph_rows], width, label="cycle rank", color="#54a24b")
    ax.set_xticks(xloc)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylabel("graph observable")
    ax.set_title("Extracted spatial-graph observables")
    ax.legend(frameon=False)
    save_paper_figure(fig, "05_static_spatial_graph_observables")
    plt.show()


### 18.5 Deformation phase-atlas templates
This is the phase-diagram use case: choose endpoint fields, scan a dense grid in `(lambda, epsilon)`, certify each level set as a compact knot tube, prune leaf artifacts from the extracted core, mildly smooth voxel stair-steps, compute Yamada polynomials on the certified core, and plot the resulting Yamada phase signatures. Invalid topology, projection failures, and boundary contact are shown as separate non-Yamada phases. Dense Yamada scans are cached under `User_guide/applications/results/04_analytic_knot_fields_cache`.


In [ ]:
PHASE_GALLERY_SPECS = [
    (
        "trefoil_to_figure8",
        lambda: KnotFunction.from_name("trefoil"),
        lambda: KnotFunction.from_name("figure_eight"),
        "knot-to-knot gauge-fixed interpolation",
    ),
]
if not QUICK:
    PHASE_GALLERY_SPECS.extend([
        (
            "trefoil_to_hopf",
            lambda: KnotFunction.from_name("trefoil"),
            lambda: KnotFunction.from_name("hopf_link"),
            "single-component to two-component endpoint comparison",
        ),
        (
            "hopf_to_cinquefoil",
            lambda: KnotFunction.from_name("hopf_link"),
            lambda: KnotFunction.from_name("cinquefoil"),
            "link-to-knot endpoint comparison",
        ),
    ])

phase_atlas_results = {}
phase_atlas_rows = []
if RUN_USE_CASE_GALLERY and RUN_GALLERY_PHASE_ATLAS:
    for label, start_factory, end_factory, claim in PHASE_GALLERY_SPECS:
        start = start_factory()
        end = end_factory()
        atlas_path = KnotFunctionPath(
            start, end,
            normalize=True,
            phase_align=True,
            sample_count=2048 if PAPER_FIGURE_MODE else (1024 if QUICK else 4096),
            seed=23,
        )
        print(f"{label}: {start.name} -> {end.name}")
        print("gauge:", atlas_path.gauge)
        phase_kind = "yamada" if RUN_YAMADA else "graph"
        atlas_result = cached_deformation_scan(
            f"{label}_{phase_kind}_certified",
            atlas_path,
            lambdas=GALLERY_LAMBDAS,
            radii=GALLERY_PHASE_RADII,
            dimension=GALLERY_PHASE_DIM,
            invariant="yamada" if RUN_YAMADA else None,
            yamada_options=YAMADA_OPTIONS if RUN_YAMADA else None,
        )
        phase_atlas_results[label] = atlas_result
        rows = phase_summary_rows(atlas_result, label)
        for row in rows:
            row["claim_template"] = claim
        phase_atlas_rows.extend(rows)
        print("records:", len(atlas_result.records))
        transition_intervals = atlas_result.transition_points()
        print_transition_summary(
            transition_intervals,
            yamada_labels=phase_label_lookup(atlas_result),
            heading="transition intervals",
            limit=40,
        )
        show_rows(rows, limit=10)

        ax, legend_rows = plot_phase_signature_grid(
            atlas_result,
            title=f"{label.replace('_', ' ')}: {phase_kind} phase signatures",
        )
        save_paper_figure(ax.figure, f"05_{label}_{phase_kind}_phase_signatures")
        plt.show()
        print_legend_rows(legend_rows)

        fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), constrained_layout=True)
        plot_phase_observable(atlas_result, "nodes", title=f"{label}: nodes", ax=axes[0])
        plot_phase_observable(atlas_result, "cycle_rank", title=f"{label}: cycle rank", ax=axes[1])
        save_paper_figure(fig, f"05_{label}_{phase_kind}_scan_graph_observable_heatmaps")
        plt.show()

show_rows(phase_atlas_rows, limit=18)


### 18.6 Optional convergence findings for selected gallery cases
Enable `RUN_GALLERY_CONVERGENCE` when a gallery cell looks scientifically interesting.  This reruns the accepted radius over multiple dimensions before you promote a static-atlas observation into a stabilized numerical claim.


In [ ]:
gallery_convergence_rows = []
if RUN_USE_CASE_GALLERY and RUN_GALLERY_CONVERGENCE:
    for label, result in gallery_diagnostic_results.items():
        accepted = [
            row for row in result["rows"]
            if row.get("matches_expected_tubular_neighborhood") is True
        ]
        if not accepted:
            gallery_convergence_rows.append({
                "use_case": label,
                "epsilon": None,
                "dimension": None,
                "converged": False,
                "error": "no accepted epsilon in static atlas",
            })
            continue
        eps = float(accepted[0]["epsilon"])
        try:
            report = result["field"].tubular_convergence(
                eps, dimensions=GALLERY_CONVERGENCE_DIMS, span=SPAN
            )
            for dimension, diagnostic in zip(report.dimensions, report.diagnostics):
                row = diagnostic_row(label, diagnostic, claim="multi-resolution follow-up")
                row.update({"dimension": int(dimension), "converged": report.converged})
                gallery_convergence_rows.append(row)
            print(label, "epsilon", eps, "converged:", report.converged)
        except Exception as exc:
            gallery_convergence_rows.append({
                "use_case": label,
                "epsilon": eps,
                "dimension": None,
                "converged": False,
                "error": f"{type(exc).__name__}: {exc}",
            })

show_rows(gallery_convergence_rows)


## 19. Paper figure export: level surfaces, spatial graphs, and phase panels
This cell produces concrete exported figures. Static panels use a higher-resolution level-set grid and overlay the extracted spatial graph on the translucent surface. The deformation strip samples more lambda values than the earlier preview; panels with boundary-contact failures remain useful numerical warnings and are represented in the Yamada phase-signature heat maps above.


In [ ]:
PAPER_STATIC_FIGURE_CASES = [
    ("Unknot T(1,1)", KnotFunction.torus(1, 1), 0.20),
    ("Trefoil T(2,3)", KnotFunction.torus(2, 3), 0.30),
    ("Hopf link T(2,2)", KnotFunction.torus(2, 2), 0.30),
    ("Cinquefoil T(2,5)", KnotFunction.torus(2, 5), 0.30),
]

paper_static_results = []
if RUN_PAPER_FIGURE_EXPORT:
    fig = plt.figure(figsize=(13.0, 3.8), constrained_layout=True)
    for index, (label, field, eps) in enumerate(PAPER_STATIC_FIGURE_CASES, start=1):
        ax = fig.add_subplot(1, len(PAPER_STATIC_FIGURE_CASES), index, projection="3d")
        result = plot_field_surface_graph_panel(
            ax, field, eps,
            title=f"{label}\n$\\epsilon={eps:.2f}$",
            span=SPAN, dimension=PAPER_SURFACE_DIM,
        )
        paper_static_results.append({
            "label": label,
            "epsilon": eps,
            "diagnostic": result["diagnostic"],
            "graph_nodes": result["graph"].number_of_nodes(),
            "graph_edges": result["graph"].number_of_edges(),
        })
    save_paper_figure(fig, "05_static_level_surfaces_with_spatial_graphs")
    plt.show()

show_rows([
    {
        "label": row["label"],
        "epsilon": row["epsilon"],
        "nodes": row["graph_nodes"],
        "edges": row["graph_edges"],
        "diagnostic_match": row["diagnostic"].matches_expected_tubular_neighborhood,
        "surface_components": row["diagnostic"].surface_components,
        "boundary_genus": row["diagnostic"].total_boundary_genus,
    }
    for row in paper_static_results
])

if RUN_PAPER_FIGURE_EXPORT:
    strip_lambdas = PAPER_STRIP_LAMBDAS
    strip_eps = 0.30
    strip_path = KnotFunctionPath(
        KnotFunction.from_name("trefoil"),
        KnotFunction.from_name("figure_eight"),
        normalize=True,
        phase_align=True,
        sample_count=2048 if PAPER_FIGURE_MODE else (1024 if QUICK else 4096),
        seed=23,
    )
    fig = plt.figure(figsize=(16.5, 3.4), constrained_layout=True)
    strip_rows = []
    for index, lam in enumerate(strip_lambdas, start=1):
        field = strip_path.at(float(lam))
        sample = field.sample(span=SPAN, dimension=PAPER_STRIP_DIM)
        ax = fig.add_subplot(1, len(strip_lambdas), index, projection="3d")
        try:
            diagnostic = field.diagnose_level(strip_eps, sample=sample, span=SPAN, dimension=PAPER_STRIP_DIM)
            mesh = field.level_surface(strip_eps, sample=sample, span=SPAN, dimension=PAPER_STRIP_DIM, require_compact=False)
            surface_pts = plot_level_surface_mesh(ax, mesh, alpha=0.34)
            graph = None
            try:
                graph = field.to_spatial_graph(strip_eps, sample=sample, span=SPAN, dimension=PAPER_STRIP_DIM)
                graph_pts = plot_spatial_graph_3d(ax, graph)
                pts = np.vstack([surface_pts, graph_pts]) if len(graph_pts) else surface_pts
            except Exception:
                pts = surface_pts
            set_axes_equal_3d(ax, pts)
            style_3d_axis(ax)
            boundary_marker = " boundary" if diagnostic.touches_box_boundary else ""
            ax.set_title(f"$\\lambda={lam:.2f}${boundary_marker}", pad=4)
            strip_rows.append({
                "lambda": float(lam),
                "epsilon": strip_eps,
                "volume_components": diagnostic.volume_components,
                "surface_components": diagnostic.surface_components,
                "euler": diagnostic.surface_euler_characteristic,
                "genus": diagnostic.total_boundary_genus,
                "touches_boundary": diagnostic.touches_box_boundary,
                "graph_nodes": None if graph is None else graph.number_of_nodes(),
                "graph_edges": None if graph is None else graph.number_of_edges(),
            })
        except Exception as exc:
            ax.text2D(0.08, 0.5, f"{type(exc).__name__}\n{exc}", transform=ax.transAxes)
            style_3d_axis(ax)
            strip_rows.append({
                "lambda": float(lam), "epsilon": strip_eps,
                "error": f"{type(exc).__name__}: {exc}",
            })
    save_paper_figure(fig, "05_deformation_level_surface_lambda_strip_highres")
    plt.show()
    show_rows(strip_rows)


## 20. Reusable arbitrary-braid analysis helper

This wraps the main workflow without hiding the numerical checks.

In [ ]:
def analyze_braid(
    word,
    *,
    strands=None,
    span=((-4.0,4.0),)*3,
    dimension=64,
    candidate_radii=(0.08,0.12,0.16,0.20,0.25),
):
    field = KnotFunction.from_braid(
        word, strands=strands, validation_samples=512
    )
    rep = field.construction_report
    print("word:", rep.word)
    print("strands:", rep.strands)
    print("permutation:", rep.permutation)
    print("closure components:", rep.components)
    print("Fourier mode:", rep.fourier_mode)
    print("validation error fraction:", rep.error_fraction)

    sample = field.sample(span=span, dimension=dimension)
    diagnostics = []
    for eps in candidate_radii:
        try:
            d = field.diagnose_level(
                eps, sample=sample, span=span, dimension=dimension
            )
            diagnostics.append(d)
            print(
                f"eps={eps:.3f}: closed={d.surface_is_closed}, "
                f"boundary={d.touches_box_boundary}, "
                f"genus={d.total_boundary_genus}, "
                f"match={d.matches_expected_tubular_neighborhood}"
            )
        except Exception as exc:
            print(f"eps={eps:.3f}: {type(exc).__name__}: {exc}")

    valid = [d for d in diagnostics if d.matches_expected_tubular_neighborhood is True]
    if not valid:
        return {"field":field, "sample":sample,
                "diagnostics":diagnostics, "graph":None}

    eps = valid[0].radius
    graph = field.to_spatial_graph(
        eps, sample=sample, span=span, dimension=dimension
    )
    return {
        "field":field, "sample":sample, "diagnostics":diagnostics,
        "epsilon":eps, "graph":graph,
    }

# Example:
# result = analyze_braid((1,-2,1,-2), dimension=GRID_DIM)


## 21. Exact/combinatorial versus numerical claims

**Exact/combinatorial for the supplied braid representation**
- parsing the Artin word;
- strand count required by that word;
- endpoint permutation;
- closure component count from permutation cycles;
- algebraic evaluation of the retained polynomial.

**Numerically validated**
- finite Fourier approximation of the geometric braid;
- finite-grid field sampling;
- marching-cubes level surface;
- voxel tubular-neighborhood topology;
- skeleton extraction;
- convergence over tested resolutions.

**Not claimed**
- computation of global knot braid index;
- proof that a supplied braid word is minimal;
- a formal proof of all analytic finite-scaling thresholds;
- a canonical path between two analytic knot representatives;
- topology correctness when the volume touches the finite box boundary or fails
  resolution convergence.

## 22. Final workflow map


$$
\begin{array}{c}
\text{named knot/link}\\
\text{torus }T(p,q)\\
\text{arbitrary Artin braid word}\\
\text{custom Python/SymPy field}
\end{array}
\longrightarrow
\text{KnotFunction}
\longrightarrow
\text{field sample}
\longrightarrow
\text{epsilon diagnostics}
\longrightarrow
\text{validated tube}
\longrightarrow
\text{spatial graph}
\longrightarrow
\text{optional invariant}.
$$


The use-case gallery above repeats this pipeline across catalogued links, supplied braid words, epsilon ranges, and gauge-fixed deformation paths. Its outputs are intended as figure and finding templates: exact braid audit tables, sampled Fourier validation tables, epsilon-acceptance maps, graph-observable heat maps, and numerical phase diagrams. A phase boundary in those plots is a finite-grid observation for the stated span, resolution, radius grid, and interpolation, not an exact critical value.
